# Days 14–15 · Labs 3 and 4 in one: joins, windows, hard patterns and performance

35 of Zillacode's 52 problems join tables, and most of the Hard ones need a window function or one of a few patterns. In real jobs, joins, windows and file layout are also where most of the time goes, so this lab covers both tracks in depth:

**Day 14 · Joins and windows**
1. Join types, and what decides the number of rows
2. 🔬 How Spark executes a join: five algorithms, how Spark picks one, hints, and AQE's switch at run time
3. Same column names, one table in two roles, self-joins
4. Joining on a range, and cross joins
5. 🔬 Skew: one key with too many rows, and three ways to fix it
6. Window functions: a group's value on every row, and how a window runs
7. Ranking: `row_number`, `rank`, `dense_rank`, top N and latest row
8. Previous and next rows, running totals and the frame trap

**Day 15 · Hard patterns and performance**

9. Patterns behind the Hard problems: sessions, streaks, the first and last row of a group, the latest snapshot, comparing with the whole table
10. Spark SQL inside `etl`, and why SQL and DataFrame code run the same plan
11. 🔬 Read less: columns, sorted files, partition folders and small files
12. 🔬 Caching: when computing once and reusing pays, and when it hurts
13. UDFs: when you need one, and what it costs (measured)
14. Debugging, and the performance mistakes that give no error

Then ⚡ **the slow report** (a final optimization problem that uses everything) and a one-page optimization checklist.

In Zillacode's topic list this is *Complex Joins* and *Windows Functions*, plus the patterns of its Hard problems. On the optimization side it covers theory Days 4 to 8: file layout and pushdown, join selection, broadcast, shuffle hash and sort-merge joins, hints, AQE's join switch and skew handling, salting, caching and UDF costs.

**Big tables.** The third setup cell generates 10 million sales (about 160 MB of Parquet, once, in about 15 seconds), because on 16 rows every join takes 0.1 seconds and nothing can be measured.


## How to work
* Run the four **Setup** cells first, then the cell that prints the small tables.
* **Lessons**: run every example cell (its output is already shown under it), then change it and run it again until you can predict the result. On the big tables your times will differ from the stored ones; the plans, stages, tasks and bytes should not.
* Exercises come right after each lesson, as on Day 13: **`etl(...)`** in Zillacode's format, **🔍 Read the plan** (answer as a comment in the cell), and **⚡ Make it faster** (the expected output *and* the target under it).
* Keep the **Spark UI** open (normally http://localhost:4040): the *SQL / DataFrame* tab shows each join's algorithm and row counts, the *Stages* tab the slowest task of every stage, and (Lesson 12) the *Storage* tab the cached data.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.
* About 3 hours for Day 14 and 2 hours 30 minutes for Day 15 (14 `etl`, 4 🔍 and 5 ⚡ in all). Lessons 1 to 8 are Day 14 of the roadmap and Lessons 9 to 14 are Day 15, so Lesson 8 is the natural place to stop for the day.

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook. The notebook writes Parquet files into a `data/` folder next to it: about 160 MB for the big tables, and Lesson 11 adds three more copies of `sales` (about 450 MB in all). Delete the folder when you're done.


## The data
The small shop tables of the earlier labs (saved as Parquet in `data/shop/`), for the `etl` exercises, and three big generated tables for the 🔬 lessons.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

**The big tables** (third setup cell), for the 🔬 lessons:
* **sales**: 10 million rows: *sale_id*, *customer_id*, *product_id*, *store_id* (1 to 200), *quantity* (1 to 5), *sale_date* (2023-01-01 to 2024-12-31), *channel* (`web`, `app` or `store`). About 140 MB.
* **customers_big**: 2 million customers: *customer_id*, *name*, *city* (10 cities), *segment* (`business` or `private`). About 18 MB, just above Spark's 10 MB broadcast limit, on purpose.
* **products_big**: 50,000 products: *product_id*, *product_name*, *category* (10 categories), *price*. Under 1 MB.

**Customer 1 is a reseller who makes 30% of all sales** (about 3 million rows). Every other customer has a handful. Lesson 5 is about what this does to a join.

## Setup
Run these cells once: Spark and the small tables, `measure()` (Day 13, Lesson 1), and the big tables, which are generated only if `data/big/` doesn't exist yet.

In [1]:
# Setup: starts Spark, creates the tables and saves them as Parquet files. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")                               # 4 cores: up to 4 tasks run at the same time
         .appName("spark-lab-14-15")
         .config("spark.driver.memory", "2g")              # the default 1 GB is tight for 10 million rows
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.driver.host", "localhost")         # run on this machine only; the Spark UI is at http://localhost:4040
         .config("spark.sql.maxMetadataStringLength", "1000")   # explain() prints long filter and column lists in full
         .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false")   # Spark 4.2: without this, the UI's "Input" misses most bytes read
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

# Save every table as Parquet and read it back. In-memory tables would show "Scan ExistingRDD" in plans;
# Parquet files show what real jobs read: "FileScan parquet" with the columns and filters it reads.
for name, df in [("customers", customers), ("products", products), ("orders", orders),
                 ("customers_new", customers_new), ("promotions", promotions),
                 ("events", events), ("referrals", referrals)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"data/shop/{name}")   # one file per table
customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")
customers_new = spark.read.parquet("data/shop/customers_new")
promotions = spark.read.parquet("data/shop/promotions")
events = spark.read.parquet("data/shop/events")
referrals = spark.read.parquet("data/shop/referrals")

print("Spark", spark.version, "is ready. Spark UI:", spark.sparkContext.uiWebUrl)
print("Tables: customers, products, orders, customers_new, promotions, events, referrals")

Spark 4.2.0 is ready. Spark UI: http://localhost:4040
Tables: customers, products, orders, customers_new, promotions, events, referrals


In [2]:
# measure(): run a query and print what it cost. These are the numbers the Spark UI shows.
import json, time, urllib.request

def ui(path):
    """Read one page of the Spark UI's REST API (the data behind http://localhost:4040)."""
    sc = spark.sparkContext
    with urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/{path}") as response:
        return json.load(response)

def size(n):
    for unit, k in (("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if n >= k:
            return f"{n / k:.1f} {unit}"
    return f"{n} B"

def measure(query, label="", stages=False, plan=False):
    """Run `query` to the end, then print: time, jobs, stages, tasks, bytes read from files, shuffled and spilled.

    query:  a DataFrame (it runs with the "noop" writer: every task runs, nothing is kept or shown),
            or a function without arguments, like  lambda: df.count()
    stages: also print one line per stage: its tasks, the median and the slowest task, its shuffle
    plan:   also print the final plan of the query, after AQE's changes (the plan in the UI's SQL tab)
    """
    sc = spark.sparkContext
    group = f"measure-{time.time_ns()}"
    sc.setJobGroup(group, label or "measure")
    start = time.perf_counter()
    try:
        if callable(query):
            query()
        else:
            query.write.format("noop").mode("overwrite").save()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    seconds = time.perf_counter() - start
    tracker = sc.statusTracker()
    jobs = sorted(tracker.getJobIdsForGroup(group))
    stage_ids = sorted({s for j in jobs for s in tracker.getJobInfo(j).stageIds})
    for _ in range(100):                     # the UI records a stage a moment after it ends
        info = [ui(f"stages/{s}")[0] for s in stage_ids]
        if all(s["status"] in ("COMPLETE", "SKIPPED", "FAILED") for s in info):
            break
        time.sleep(0.05)
    ran = [s for s in info if s["status"] == "COMPLETE"]       # "SKIPPED": output reused, did not run
    total = lambda key: sum(s[key] for s in ran)
    print(f"{label + ': ' if label else ''}{seconds:.2f} s | {len(jobs)} jobs, {len(ran)} stages, "
          f"{total('numCompleteTasks')} tasks | read {size(total('inputBytes'))} | "
          f"shuffle {size(total('shuffleWriteBytes'))} | spill {size(total('diskBytesSpilled'))}")
    if stages:
        for s in ran:
            t = ui(f"stages/{s['stageId']}/{s['attemptId']}/taskSummary?quantiles=0.5,1.0")["executorRunTime"]
            print(f"   stage {s['stageId']:>3}: {s['numCompleteTasks']:>3} tasks, median {t[0] / 1000:.2f} s, "
                  f"max {t[1] / 1000:.2f} s | read {size(s['inputBytes'])} | "
                  f"shuffle read {size(s['shuffleReadBytes'])}, write {size(s['shuffleWriteBytes'])}")
    if plan:
        runs = [e for e in ui("sql?offset=0&length=1000000") if set(e["successJobIds"]) & set(jobs)]
        tree = ui(f"sql/{runs[-1]['id']}?planDescription=true")["planDescription"].split("\n\n")[0]
        print(tree.replace(", Statistics(sizeInBytes=8.0 EiB)", ""))

print("measure() is ready")

measure() is ready


In [3]:
# The big tables: generated once into data/big/ (about 15 seconds), then only read
import os
if not os.path.exists("data/big/sales/_SUCCESS"):
    spark.range(1, 10_000_001, numPartitions=8).select(
        F.col("id").alias("sale_id"),
        # customer 1 (a reseller) gets 30% of the sales; the rest are spread over customers 2 to 2,000,000
        F.when(F.rand(1) < 0.3, 1).otherwise((F.rand(2) * 1_999_999).cast("int") + 2).alias("customer_id"),
        ((F.rand(3) * 50_000).cast("int") + 1).alias("product_id"),
        ((F.rand(4) * 200).cast("int") + 1).alias("store_id"),
        ((F.rand(5) * 5).cast("int") + 1).alias("quantity"),
        F.date_add(F.lit("2023-01-01").cast("date"), (F.rand(6) * 731).cast("int")).alias("sale_date"),
        F.element_at(F.array(F.lit("web"), F.lit("app"), F.lit("store")), (F.rand(7) * 3).cast("int") + 1).alias("channel"),
    ).write.mode("overwrite").parquet("data/big/sales")
if not os.path.exists("data/big/customers/_SUCCESS"):
    cities = ["Tashkent", "Samarkand", "Bukhara", "Berlin", "Munich", "Madrid", "London", "Paris", "Rome", "Osaka"]
    spark.range(1, 2_000_001, numPartitions=4).select(
        F.col("id").cast("int").alias("customer_id"),
        F.concat(F.lit("Customer "), F.col("id")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in cities]), (F.col("id") % 10).cast("int") + 1).alias("city"),
        F.when(F.col("id") % 7 == 0, "business").otherwise("private").alias("segment"),
    ).write.mode("overwrite").parquet("data/big/customers")
if not os.path.exists("data/big/products/_SUCCESS"):
    categories = ["Books", "Electronics", "Kitchen", "Home", "Toys", "Garden", "Sports", "Beauty", "Music", "Games"]
    spark.range(1, 50_001, numPartitions=1).select(
        F.col("id").cast("int").alias("product_id"),
        F.concat(F.lit("Product "), F.col("id")).alias("product_name"),
        F.element_at(F.array(*[F.lit(c) for c in categories]), (F.col("id") % 10).cast("int") + 1).alias("category"),
        F.round(F.rand(8) * 500 + 5, 2).alias("price"),
    ).write.mode("overwrite").parquet("data/big/products")

sales = spark.read.parquet("data/big/sales")
customers_big = spark.read.parquet("data/big/customers")
products_big = spark.read.parquet("data/big/products")
print(f"sales: {sales.count():,} rows | customers_big: {customers_big.count():,} | products_big: {products_big.count():,}")
sales.show(3)

sales: 10,000,000 rows | customers_big: 2,000,000 | products_big: 50,000
+-------+-----------+----------+--------+--------+----------+-------+
|sale_id|customer_id|product_id|store_id|quantity| sale_date|channel|
+-------+-----------+----------+--------+--------+----------+-------+
|2500001|          1|      1196|      15|       2|2023-12-20|  store|
|2500002|    1906291|      4755|     148|       3|2024-06-20|    app|
|2500003|     132273|     38526|      97|       4|2023-07-05|    web|
+-------+-----------+----------+--------+--------+----------+-------+
only showing top 3 rows


Print every table, so you can check results by eye:

In [4]:
for name in ["customers", "products", "orders", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 1 · Join types

`left.join(right, on, how)`:

| `how` | Keeps |
|---|---|
| `"inner"` (the default) | the rows that have a match on both sides |
| `"left"` (also `"left_outer"`) | every left row; the right columns are null where nothing matched |
| `"right"` | every right row |
| `"full"` (also `"outer"`, `"full_outer"`) | every row of both sides |
| `"left_semi"` | the left rows that **have** a match: only the left columns, and never duplicated (SQL `WHERE EXISTS`) |
| `"left_anti"` | the left rows that have **no** match: only the left columns (SQL `WHERE NOT EXISTS`) |
| `a.crossJoin(b)` | every combination: rows of a × rows of b (Lesson 4) |

**Three ways to write `on`:**

| `on=` | Result |
|---|---|
| `"customer_id"` | the same name on both sides; the output has **one** customer_id column |
| `["customer_id", "day"]` | several keys with the same names |
| `orders.customer_id == customers.customer_id`, or with different names `mines.id == extraction.mine_id` | an expression; the output keeps **both** key columns (Lesson 3) |

**What decides the number of rows:**
* **A null key never matches**, not even another null. The guest order (no customer_id) drops out of every inner join on customer_id.
* **One-to-many repeats rows**: a customer with 4 orders appears 4 times after the join (and Day 13, Lesson 6: two "many" tables multiply).
* **After a full join** on an expression, the key of the side without a match is null: combine with `F.coalesce(a.key, b.key)`. With `on="key"`, Spark does that for you.

On Zillacode, an "all X, even without Y" sentence means a left join; "only X that have Y" means inner or left_semi; "X that never..." means left_anti; "all records from both" means full (#14, #24, #47). [join docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html).

In [5]:
c = customers.select("customer_id", "name")
o = orders.select("order_id", "customer_id", "status")
for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]:
    print(f"{how:10}", c.join(o, "customer_id", how).count(), "rows")

inner      15 rows
left       16 rows
right      16 rows
full       17 rows
left_semi  6 rows
left_anti  1 rows


In [6]:
# left: Grace (7) has no orders, so her order columns are null. right: the guest order has no customer.
c.join(o, "customer_id", "left").filter(F.col("customer_id") >= 6).orderBy("customer_id").show()
c.join(o, "customer_id", "right").filter(F.col("customer_id").isNull()).show()

+-----------+------------+--------+-------+
|customer_id|        name|order_id| status|
+-----------+------------+--------+-------+
|          6|Farid Aliyev|    1012|shipped|
|          7|   Grace Lee|    NULL|   NULL|
+-----------+------------+--------+-------+

+-----------+----+--------+---------+
|customer_id|name|order_id|   status|
+-----------+----+--------+---------+
|       NULL|NULL|    1011|delivered|
+-----------+----+--------+---------+



In [7]:
# semi and anti: only the left columns, at most one row per left row
c.join(o, "customer_id", "left_semi").orderBy("customer_id").show()
c.join(o, "customer_id", "left_anti").show()

+-----------+----------------+
|customer_id|            name|
+-----------+----------------+
|          1|    Aziz Karimov|
|          2|   Bella Schmidt|
|          3|     Carlos Ruiz|
|          4|Dilnoza Rahimova|
|          5|     Emma Wilson|
|          6|    Farid Aliyev|
+-----------+----------------+

+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+



### Exercises

**Q1 · `etl(customers, orders)`** Which customers have never placed an order? Show customer_id and name.

Expected output:

```
+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q2 · `etl(customers, orders)`** Show every customer with their number of orders, including customers with 0 orders. Show name and orders, most orders first; on a tie, by name.

Expected output:

```
+----------------+------+
|            name|orders|
+----------------+------+
|    Aziz Karimov|     4|
|   Bella Schmidt|     3|
|     Carlos Ruiz|     3|
|Dilnoza Rahimova|     2|
|     Emma Wilson|     2|
|    Farid Aliyev|     1|
|       Grace Lee|     0|
+----------------+------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q3 · `etl(orders, customers)`** Which customers had all of their orders delivered (and at least one order)? Show their names, from A to Z.

Expected output:

```
+-------------+
|         name|
+-------------+
|Bella Schmidt|
+-------------+
```

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

## Lesson 2 · 🔬 How Spark executes a join

Rows that match must meet in the same task. Spark has five algorithms for that (theory Day 5), and each one leaves its own shape in the plan:

| Algorithm | How it works | In the plan | Cost |
|---|---|---|---|
| **Broadcast hash join** | the small side is collected to the driver and a copy goes to every task, which builds a hash table from it; each task then streams its part of the big side through that table | `BroadcastHashJoin`, with a `BroadcastExchange` on the small side and **no `Exchange` on the big side** | the big side isn't moved at all: the cheapest join, as long as the small side fits in memory |
| **Shuffle hash join** | both sides are shuffled by the key; each task builds a hash table from its part of one side | `ShuffledHashJoin`, an `Exchange` on each side, no `Sort` | two shuffles; one side's partition must fit in memory as a hash table |
| **Sort-merge join** | both sides are shuffled by the key and sorted; each task walks the two sorted lists side by side | `SortMergeJoin`, with `Exchange` and `Sort` on each side | two shuffles and two sorts, but sorting can spill to disk, so it rarely runs out of memory: the default for two big tables |
| **Broadcast nested loop join** | one side is broadcast, and every row of the other side is compared with **every** broadcast row | `BroadcastNestedLoopJoin` | rows × rows comparisons: what you get without an `=` in the condition (range joins, cross joins) |
| **Cartesian product** | every partition of one side with every partition of the other | `CartesianProduct` | the same comparisons, without the broadcast: the most expensive |

**How Spark picks one** ([hints in the tuning guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html#join-strategy-hints); for the curious, `JoinSelection` in [SparkStrategies.scala](https://github.com/apache/spark/blob/master/sql/core/src/main/scala/org/apache/spark/sql/execution/SparkStrategies.scala)):
1. A **hint** wins, if that algorithm can run this join: `F.broadcast(df)` (or `df.hint("broadcast")`), `df.hint("merge")`, `df.hint("shuffle_hash")`, `df.hint("shuffle_replicate_nl")`.
2. With an **equality** in the condition: **broadcast** a side whose *estimated* size is under `spark.sql.autoBroadcastJoinThreshold` (10 MB) and which this join type can broadcast (next table); otherwise **sort-merge**. (Without a hint, shuffle hash is chosen only if you turn `spark.sql.join.preferSortMergeJoin` off.)
3. With **no** equality: broadcast nested loop when a side can be broadcast, otherwise a Cartesian product.

**Which side can be broadcast?** Not a side whose every row must appear in the result: a hash table doesn't tell Spark which of its rows never matched.

| Join type | Can broadcast |
|---|---|
| inner | either side |
| `left`, `left_semi`, `left_anti` | the **right** side only |
| `right` | the **left** side only |
| `full` | neither: a full join always shuffles |

So `F.broadcast(customers).join(orders, "customer_id", "left")` ignores the hint (example below).

**Estimates, not facts.** The planner estimates each side from its files: a Parquet table's size on disk, scaled down to the columns the query uses (`explain(mode="cost")` prints the estimates). It doesn't know how many rows a filter keeps, unless you've computed table statistics, so a filtered table looks as big as the whole file. **AQE corrects this at run time**: when the shuffle of a sort-merge join has run, it knows the real size of each side, and if one turned out small, it **switches to a broadcast hash join** ([docs](https://spark.apache.org/docs/latest/sql-performance-tuning.html#converting-sort-merge-join-to-broadcast-join); in the final plan, the big side reads its shuffle files with `AQEShuffleRead local`, without the network). The switch saves the sort and the network, but the big side has already been shuffled to disk by then. When you know a side is small (a filtered or aggregated dimension), `F.broadcast(...)` saves that shuffle as well.

In [11]:
# Small tables: products (under 10 MB) is broadcast, orders is not moved
orders.join(products, "product_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#43, order_id#41, customer_id#42, quantity#44, order_date#45, status#46, product_name#38, category#39, price#40]
   +- BroadcastHashJoin [product_id#43], [product_id#37], Inner, BuildRight, false, false
      :- Filter isnotnull(product_id#43)
      :  +- FileScan parquet [order_id#41,customer_id#42,product_id#43,quantity#44,order_date#45,status#46] Batched: true, DataFilters: [isnotnull(product_id#43)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=1661]
         +- Filter isnotnull(product_id#37)
            +- FileScan parquet [product_id#37,product_name#38,category#39,price#

In [12]:
# With broadcasting switched off, the same join becomes a sort-merge join: Exchange + Sort on each side
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
orders.join(products, "product_id").explain()
# The shuffle_hash hint: an Exchange on each side, but no Sort
orders.join(products.hint("shuffle_hash"), "product_id").explain()
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#43, order_id#41, customer_id#42, quantity#44, order_date#45, status#46, product_name#38, category#39, price#40]
   +- SortMergeJoin [product_id#43], [product_id#37], Inner
      :- Sort [product_id#43 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(product_id#43, 200), ENSURE_REQUIREMENTS, [plan_id=1691]
      :     +- Filter isnotnull(product_id#43)
      :        +- FileScan parquet [order_id#41,customer_id#42,product_id#43,quantity#44,order_date#45,status#46] Batched: true, DataFilters: [isnotnull(product_id#43)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
      +- Sort [product_id#37 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(product_id#37, 200)

In [13]:
# A left join can only broadcast its right side: the hint on customers is ignored, orders is broadcast instead
F.broadcast(customers).join(orders, "customer_id", "left").explain()
# A full join can't broadcast either side: sort-merge, even for 7 and 16 rows
customers.join(orders, "customer_id", "full").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#32, name#33, email#34, city#35, signup_date#36, order_id#41, product_id#43, quantity#44, order_date#45, status#46]
   +- BroadcastHashJoin [customer_id#32], [customer_id#42], LeftOuter, BuildRight, false, false
      :- FileScan parquet [customer_id#32,name#33,email#34,city#35,signup_date#36] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/customers], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int,name:string,email:string,city:string,signup_date:date>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[1, int, false] as bigint)),false), [plan_id=1750]
         +- Filter isnotnull(customer_id#42)
            +- FileScan parquet [order_id#41,customer_id#42,product_id#43,quantity#44,order_date#45,status#46] Batched: true, DataFilters: [isnotnull(customer_id#42)], Format: Parquet, Location: 

In [14]:
# Big tables: customers_big is about 18 MB, over the 10 MB limit, so Spark plans a sort-merge join
joined = sales.join(customers_big, "customer_id")
joined.explain()
measure(joined, "sort-merge join", stages=True)
# Broadcasting customers_big anyway: the 10 million sales stay where they are
measure(sales.join(F.broadcast(customers_big), "customer_id"), "broadcast join", stages=True)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#82, sale_id#81L, product_id#83, store_id#84, quantity#85, sale_date#86, channel#87, name#89, city#90, segment#91]
   +- SortMergeJoin [customer_id#82], [customer_id#88], Inner
      :- Sort [customer_id#82 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(customer_id#82, 200), ENSURE_REQUIREMENTS, [plan_id=1803]
      :     +- Filter isnotnull(customer_id#82)
      :        +- FileScan parquet [sale_id#81L,customer_id#82,product_id#83,store_id#84,quantity#85,sale_date#86,channel#87] Batched: true, DataFilters: [isnotnull(customer_id#82)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:date,channel:string>
      +- Sort [customer_id#88 ASC NULLS FIRST], false, 0
         +- Exchange 

In [15]:
# One day of sales: the planner estimates the filtered side as big as the whole table...
last_day = (sales.filter(F.col("sale_date") == "2024-12-31")
                 .join(customers_big, "customer_id")
                 .select("sale_id", "name", "city", "segment"))
last_day.explain(mode="cost")
# ...so it plans a sort-merge join; at run time AQE sees about 14,000 rows and switches to a broadcast join
measure(last_day, "last day x customers", plan=True)

== Optimized Logical Plan ==
Project [sale_id#81L, name#89, city#90, segment#91], Statistics(sizeInBytes=751.4 TiB)
+- Join Inner, (customer_id#82 = customer_id#88), Statistics(sizeInBytes=830.5 TiB)
   :- Project [sale_id#81L, customer_id#82], Statistics(sizeInBytes=47.9 MiB)
   :  +- Filter ((isnotnull(sale_date#86) AND (sale_date#86 = 2024-12-31)) AND isnotnull(customer_id#82)), Statistics(sizeInBytes=134.3 MiB)
   :     +- Relation [sale_id#81L,customer_id#82,product_id#83,store_id#84,quantity#85,sale_date#86,channel#87] parquet, Statistics(sizeInBytes=134.3 MiB)
   +- Filter isnotnull(customer_id#88), Statistics(sizeInBytes=17.3 MiB)
      +- Relation [customer_id#88,name#89,city#90,segment#91] parquet, Statistics(sizeInBytes=17.3 MiB)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [sale_id#81L, name#89, city#90, segment#91]
   +- SortMergeJoin [customer_id#82], [customer_id#88], Inner
      :- Sort [customer_id#82 ASC NULLS FIRST], false, 0
      :  +- Exchan

### Exercises

**Q4 · 🔍 Read the plan** **Predict** the algorithm of each join, and which side is broadcast (if any). Then run the cell and check.

In [ ]:
# (a)
orders.join(customers, "customer_id", "left").explain()
# (b)
orders.join(customers, "customer_id", "left_anti").explain()
# (c)
customers_new.join(customers, "customer_id", "full").explain()
# (d)
products.join(promotions, products.price > promotions.discount_pct * 10).explain()
# Your answers:

**Q5 · ⚡ `etl(sales, customers_big)`** For every city and segment, count the sales and add up the items, over all 10 million sales. Sort by city, then segment. `slow_etl` gets it right with a sort-merge join; make Spark skip the big shuffle.

Expected output:

```
+---------+--------+-------+--------+
|     city| segment|  sales|   items|
+---------+--------+-------+--------+
|   Berlin|business| 100333|  300332|
|   Berlin| private| 599779| 1797005|
|  Bukhara|business|  99825|  300374|
|  Bukhara| private| 600575| 1802763|
|   London|business| 100257|  301404|
|   London| private| 598971| 1796369|
|   Madrid|business|  99516|  298603|
|   Madrid| private| 600367| 1801061|
|   Munich|business| 100401|  302030|
|   Munich| private| 599394| 1797227|
|    Osaka|business|  99878|  300327|
|    Osaka| private| 600443| 1802890|
|    Paris|business| 100229|  300910|
|    Paris| private| 600763| 1801993|
|     Rome|business| 100076|  300206|
|     Rome| private| 600013| 1800734|
|Samarkand|business| 100030|  299900|
|Samarkand| private|3597732|10793985|
| Tashkent|business| 100215|  300142|
| Tashkent| private| 601203| 1806120|
+---------+--------+-------+--------+
```

**Target:** a `BroadcastHashJoin` with no `Exchange` on the sales side, and a few KB of shuffle in `measure` instead of about 100 MB.

In [ ]:
def slow_etl(sales, customers_big):
    return (sales.join(customers_big, "customer_id")
                 .groupBy("city", "segment")
                 .agg(F.count("*").alias("sales"), F.sum("quantity").alias("items"))
                 .orderBy("city", "segment"))

def etl(sales, customers_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, customers_big)

measure(slow_etl(sales, customers_big), "slow")
measure(etl(sales, customers_big), "yours")
etl(sales, customers_big).show()
etl(sales, customers_big).explain()

## Lesson 3 · Same column names, one table in two roles, self-joins

After a join on an expression, the columns of **both** sides are there. If both sides have a column with the same name (`id`, `name`, `customer_id`), the plain name is **ambiguous**, and Spark stops with `[AMBIGUOUS_REFERENCE]`. Three fixes:

1. **Pick through the DataFrame**: `select(stars["name"].alias("star_name"), planets["name"].alias("planet_name"))`. Zillacode's #33 and #52 do this.
2. **Alias the DataFrames** and use dotted names: `a = df.alias("a")`, then `F.col("a.name")`. #27 and #42 do this. It's the only way in a **self-join**, where both sides are the same DataFrame.
3. **Rename before the join**, so there is nothing to confuse: `airports.withColumnRenamed("airport_name", "origin_name")`. #49 does this. It's the cleanest choice when one table plays **two roles**.

**One table in two roles** (origin and destination airport; the customer who invited and the customer who joined) means two joins with two renamed copies of the same table.

**A self-join** compares rows of one table with each other: pairs in the same city, employees and their managers, an order and the same customer's earlier orders. To get each pair once, and never pair a row with itself, add `a.id < b.id` to the condition.

In [18]:
# Ambiguous: both sides have customer_id after a join on an expression
try:
    c.join(o, c.customer_id == o.customer_id).select("customer_id").show()
except Exception as e:
    print(type(e).__name__, "->", str(e).split(".")[0])
# Fix 1: say which side you mean
c.join(o, c.customer_id == o.customer_id).select(c["customer_id"], "name", "order_id").orderBy("order_id").show(3)

AnalysisException -> [AMBIGUOUS_REFERENCE] Reference `customer_id` is ambiguous, could be: [`customer_id`, `customer_id`]
+-----------+-------------+--------+
|customer_id|         name|order_id|
+-----------+-------------+--------+
|          1| Aziz Karimov|    1001|
|          1| Aziz Karimov|    1002|
|          2|Bella Schmidt|    1003|
+-----------+-------------+--------+
only showing top 3 rows


In [19]:
# One table in two roles: rename it once per role, then join twice
airports = spark.createDataFrame([("TAS", "Tashkent"), ("IST", "Istanbul"), ("BER", "Berlin")], "code string, city string")
flights = spark.createDataFrame([(1, "TAS", "IST"), (2, "IST", "BER"), (3, "BER", "TAS")], "flight_id int, origin string, destination string")
origin = airports.select(F.col("code").alias("origin"), F.col("city").alias("origin_city"))
destination = airports.select(F.col("code").alias("destination"), F.col("city").alias("destination_city"))
(flights.join(origin, "origin")
        .join(destination, "destination")
        .select("flight_id", "origin_city", "destination_city")
        .orderBy("flight_id")
        .show())

+---------+-----------+----------------+
|flight_id|origin_city|destination_city|
+---------+-----------+----------------+
|        1|   Tashkent|        Istanbul|
|        2|   Istanbul|          Berlin|
|        3|     Berlin|        Tashkent|
+---------+-----------+----------------+



In [20]:
# Self-join: pairs of products in the same category, each pair once
a, b = products.alias("a"), products.alias("b")
(a.join(b, (F.col("a.category") == F.col("b.category")) & (F.col("a.product_id") < F.col("b.product_id")))
  .select(F.col("a.category").alias("category"),
          F.col("a.product_name").alias("product_a"),
          F.col("b.product_name").alias("product_b"))
  .orderBy("category")
  .show())

+-----------+------------+----------+
|   category|   product_a| product_b|
+-----------+------------+----------+
|      Books|       Novel|  Cookbook|
|Electronics|      Laptop|Headphones|
|    Kitchen|Coffee Maker|   Blender|
+-----------+------------+----------+



### Exercises

**Q6 · `etl(referrals, customers)`** Show every referral with both names: referral_date, referrer (the name of the customer who invited) and referred (the name of the customer who joined; empty if unknown). Sort by referral_date.

Expected output:

```
+-------------+----------------+----------------+
|referral_date|        referrer|        referred|
+-------------+----------------+----------------+
|   2024-02-10|    Aziz Karimov|Dilnoza Rahimova|
|   2024-02-25|Dilnoza Rahimova|     Emma Wilson|
|   2024-03-08|    Aziz Karimov|    Farid Aliyev|
|   2024-03-12|   Bella Schmidt|       Grace Lee|
|   2024-03-30|   Bella Schmidt|            NULL|
+-------------+----------------+----------------+
```

In [ ]:
def etl(referrals, customers):
    # Write code here: return a DataFrame
    return referrals

etl(referrals, customers).show()

**Q7 · `etl(customers)`** Show the pairs of customers who live in the same city: city, customer_a and customer_b (names). Each pair once, customer_a being the one with the smaller customer_id. Sort by city.

Expected output:

```
+--------+-------------+----------------+
|    city|   customer_a|      customer_b|
+--------+-------------+----------------+
|  Berlin|Bella Schmidt|       Grace Lee|
|Tashkent| Aziz Karimov|Dilnoza Rahimova|
+--------+-------------+----------------+
```

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

## Lesson 4 · Joining on a range, and cross joins

A join condition can be any true/false expression, not only `==`:
* **A date inside a period**: `F.col("order_date").between(F.col("start_date"), F.col("end_date"))`, usually together with an equality (`products.category == promotions.category`). Zillacode's #22 (minutes watched while a subscription was active) is this pattern.
* **A value inside a band**: price bands, tax brackets, age groups.

A join on equality plus a range is fast. A join on a range alone compares every pair of rows, which is fine for small tables like Zillacode's.

`crossJoin` pairs every row of one side with every row of the other (Zillacode #38). Its typical job is to **build a complete grid** (every customer × every category, every day × every store), then `left` join the real numbers onto it and fill the gaps with 0. That way, combinations without any data still appear in the answer.

🔬 **In the plan.** A condition without `=` leaves Spark only the nested-loop algorithms (Lesson 2): `BroadcastNestedLoopJoin` when a side can be broadcast, `CartesianProduct` otherwise. Both compare every pair of rows: 10 million sales × 1,000 promotions would be 10 billion comparisons. **Add an equality whenever the data allows it**, like `category` in the next cell: Spark then runs a hash or sort-merge join on the equal part and checks the range only for the pairs that already match (the range appears as the join's extra condition).

In [23]:
bands = spark.createDataFrame([("budget", 0.0, 49.99), ("standard", 50.0, 199.99), ("premium", 200.0, 99999.0)],
                              "band string, low double, high double")
(products.join(bands, F.col("price").between(F.col("low"), F.col("high")))
         .select("product_name", "price", "band")
         .orderBy("price")
         .show())

+------------+------+--------+
|product_name| price|    band|
+------------+------+--------+
|       Novel|  15.0|  budget|
|    Cookbook|  25.0|  budget|
|   Desk Lamp|  40.0|  budget|
|     Blender|  60.0|standard|
|Coffee Maker|  85.0|standard|
|  Headphones| 150.0|standard|
|      Laptop|1200.0| premium|
+------------+------+--------+



In [24]:
# A full grid of month × status, then the real counts, then 0 for the gaps
months = orders.select(F.month("order_date").alias("month")).distinct()
statuses = orders.select("status").distinct()
counts = orders.groupBy(F.month("order_date").alias("month"), "status").agg(F.count("*").alias("orders"))
(months.crossJoin(statuses)
       .join(counts, ["month", "status"], "left")
       .fillna(0)
       .orderBy("month", "status")
       .show())

+-----+---------+------+
|month|   status|orders|
+-----+---------+------+
|    1|cancelled|     0|
|    1|delivered|     3|
|    1|  shipped|     0|
|    2|cancelled|     1|
|    2|delivered|     2|
|    2|  shipped|     1|
|    3|cancelled|     1|
|    3|delivered|     5|
|    3|  shipped|     3|
+-----+---------+------+



In [25]:
# Range only: a nested loop, every order compared with every promotion
orders.join(promotions, orders.order_date.between(promotions.start_date, promotions.end_date)).explain()
# Equality + range: a hash join on category; the dates are checked only for pairs with the same category
with_category = orders.join(products, "product_id")
cond = (with_category.category == promotions.category) & with_category.order_date.between(promotions.start_date, promotions.end_date)
with_category.join(promotions, cond).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastNestedLoopJoin BuildRight, Inner, ((order_date#45 >= start_date#54) AND (order_date#45 <= end_date#55))
   :- Filter isnotnull(order_date#45)
   :  +- FileScan parquet [order_id#41,customer_id#42,product_id#43,quantity#44,order_date#45,status#46] Batched: true, DataFilters: [isnotnull(order_date#45)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(order_date)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
   +- BroadcastExchange IdentityBroadcastMode, [plan_id=4081]
      +- Filter (isnotnull(start_date#54) AND isnotnull(end_date#55))
         +- FileScan parquet [promo_id#52,category#53,start_date#54,end_date#55,discount_pct#56] Batched: true, DataFilters: [isnotnull(start_date#54), isnotnull(end_date#55)], Format: Parquet, Location: InMemoryFileIndex(1 paths

### Exercise

**Q8 · `etl(orders, products, promotions)`** For each delivered order, find the promotion of its product's category that was running on the order date. Show order_id, product_name, promo_id and discount_pct; orders without a promotion drop out. Sort by order_id.

Expected output:

```
+--------+------------+--------+------------+
|order_id|product_name|promo_id|discount_pct|
+--------+------------+--------+------------+
|    1001|      Laptop|  WINTER|          10|
|    1003|  Headphones|  WINTER|          10|
|    1009|       Novel|BOOKWEEK|          20|
|    1011|     Blender|  SPRING|          15|
|    1013|      Laptop|   FLASH|           5|
+--------+------------+--------+------------+
```

In [ ]:
def etl(orders, products, promotions):
    # Write code here: return a DataFrame
    return orders

etl(orders, products, promotions).show()

## Lesson 5 · 🔬 Skew: when one key has too many rows

In `sales`, customer 1 (the reseller) has 3 million of the 10 million rows. A shuffle sends all rows of a key to the **same** partition, so one task gets 3 million rows while every other task gets about 35,000. A stage lasts as long as its slowest task, and that task may also run short of memory and spill to disk (theory Day 6).

**How to see skew**
* In the data: count the rows per key (`groupBy(key).count()`, most first), or the rows per partition after a shuffle with `F.spark_partition_id()` (first cell below).
* In the Spark UI: open the join's stage in the *Stages* tab. In *Summary Metrics*, the **Max** duration and shuffle read are far above the **Median**. `measure(..., stages=True)` prints the median and the slowest task of every stage.

**What suffers from a hot key**
* **Joins** that shuffle (sort-merge, shuffle hash): all rows of the key meet in one task.
* **Windows** partitioned by the key: one task sorts and scans the whole partition, and AQE can't split it (Q9).
* **Not `groupBy` with `sum`, `count`, `avg`...**: partial aggregation (Day 13, Lesson 5) sends one row per key per task, so the hot key costs almost nothing extra. (`collect_list` of the hot key does build one huge list.)

**Fixes for a skewed join, best first**
1. **Broadcast the other side** when it's small enough: no shuffle, so no skew (Lesson 2).
2. **AQE's skew join** (on by default, [docs](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-skew-join)): after the shuffle, AQE looks for partitions bigger than 5 × the median (`spark.sql.adaptive.skewJoin.skewedPartitionFactor`) **and** bigger than 256 MB (`skewedPartitionThresholdInBytes`). It splits such a partition into pieces of about 64 MB (`advisoryPartitionSizeInBytes`) and joins each piece with a copy of the other side's matching partition. The final plan shows `SortMergeJoin(skew=true)` and `AQEShuffleRead skewed`. It works for sort-merge and shuffle hash joins (for left and right joins, on the side whose rows are all kept). Two things to know: on a laptop our hot partition is under 256 MB, so the examples lower the thresholds; and AQE measures partitions in **compressed shuffle bytes**, not rows. Customer 1's partition is about 70 MB when every column is shuffled, but only about 10 MB when just `customer_id` and `quantity` are: then AQE sees no skew, while that task still processes 3 million rows.
3. **Salting**, when AQE can't help (an older Spark, a full join, a hot key that compresses well): give the big side's rows of the hot key a random number 0..N-1 (the *salt*), copy the other side's row for that key N times (one per salt), and join on (key, salt). The hot key's rows now spread over N partitions. Salting only the hot keys keeps the copies small; salting every key copies the whole other side N times.
4. **Split the work**: join the hot key's rows with a broadcast join, the rest with a normal join, and `union` the two results.

In [27]:
# Rows per shuffle partition when the sales are hash-partitioned by customer_id (200 partitions)
per_partition = (sales.repartition(200, "customer_id")
                      .groupBy(F.spark_partition_id().alias("partition")).count())
per_partition.agg(F.min("count").alias("min_rows"),
                  F.percentile("count", 0.5).alias("median_rows"),
                  F.max("count").alias("max_rows")).show()

+--------+-----------+--------+
|min_rows|median_rows|max_rows|
+--------+-----------+--------+
|   33885|    35008.5| 3032973|
+--------+-----------+--------+



In [28]:
# The sort-merge join with AQE's skew join switched off. Coalescing is off too, so each partition is one task
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
measure(sales.join(customers_big, "customer_id"), "skewed join", stages=True)
# AQE's skew join on, with laptop-sized thresholds (the defaults are 256 MB and 64 MB)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "16MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "16MB")
measure(sales.join(customers_big, "customer_id"), "AQE skew join", stages=True, plan=True)
for key in ["spark.sql.adaptive.coalescePartitions.enabled", "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    spark.conf.unset(key)                        # back to the defaults

skewed join: 3.13 s | 3 jobs, 3 stages, 208 tasks | read 160.4 MB | shuffle 305.7 MB | spill 48.3 MB
   stage 170:   4 tasks, median 0.38 s, max 0.38 s | read 18.4 MB | shuffle read 0 B, write 34.3 MB
   stage 171:   4 tasks, median 0.94 s, max 0.94 s | read 142.1 MB | shuffle read 0 B, write 271.5 MB
   stage 174: 200 tasks, median 0.02 s, max 1.34 s | read 0 B | shuffle read 305.7 MB, write 0 B
AQE skew join: 2.55 s | 3 jobs, 3 stages, 211 tasks | read 160.4 MB | shuffle 305.7 MB | spill 0 B
   stage 175:   4 tasks, median 0.34 s, max 0.34 s | read 18.4 MB | shuffle read 0 B, write 34.3 MB
   stage 176:   4 tasks, median 0.96 s, max 0.97 s | read 142.1 MB | shuffle read 0 B, write 271.5 MB
   stage 179: 203 tasks, median 0.01 s, max 0.25 s | read 0 B | shuffle read 306.2 MB, write 0 B
== Physical Plan ==
OverwriteByExpression noop-table (27)
+- AdaptiveSparkPlan (26)
   +- == Final Plan ==
      ResultQueryStage (17)
      +- * Project (16)
         +- * SortMergeJoin(skew=true) Inne

In [29]:
# A groupBy on the hot key is not a problem: partial aggregation sends one row per key from each task
measure(sales.groupBy("customer_id").agg(F.sum("quantity").alias("items")), "groupBy customer_id", stages=True)

groupBy customer_id: 1.37 s | 2 jobs, 2 stages, 8 tasks | read 36.5 MB | shuffle 46.9 MB | spill 0 B
   stage 180:   4 tasks, median 0.97 s, max 0.97 s | read 36.5 MB | shuffle read 0 B, write 46.9 MB
   stage 182:   4 tasks, median 0.36 s, max 0.36 s | read 0 B | shuffle read 46.9 MB, write 0 B


### Exercises

**Q9 · 🔍 Read the plan** This window numbers each customer's sales by date. Run it. Which stage is skewed, and by how much (median vs slowest task)? Why doesn't AQE fix it, and what would you do if you had to run it every day?

In [ ]:
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")   # one task per partition
nth = W.partitionBy("customer_id").orderBy("sale_date", "sale_id")
measure(sales.withColumn("nth_purchase", F.row_number().over(nth)), "window by customer", stages=True)
spark.conf.unset("spark.sql.adaptive.coalescePartitions.enabled")
# Your answer:

**Q10 · ⚡ `etl(sales, customers_big)`** Add up the items of each city, sorted by city. In this cell Spark can't broadcast and AQE's skew join is off, so `slow_etl` runs a sort-merge join where one task gets the reseller's 3 million rows. Balance it with salting: spread customer 1's sales over 16 salt values.

Expected output:

```
+---------+--------+
|     city|   items|
+---------+--------+
|   Berlin| 2097337|
|  Bukhara| 2103137|
|   London| 2097773|
|   Madrid| 2099664|
|   Munich| 2099257|
|    Osaka| 2103217|
|    Paris| 2102903|
|     Rome| 2100940|
|Samarkand|11093885|
| Tashkent| 2106262|
+---------+--------+
```

**Target:** in the join stage (200 tasks, about 95 MB of shuffle read), the slowest task (`max`) is at least 2 times faster than with `slow_etl` (3 to 8 times is typical).

In [ ]:
# Two big tables: broadcasting and AQE's skew join are switched off for this exercise
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

def slow_etl(sales, customers_big):
    return (sales.join(customers_big, "customer_id")
                 .groupBy("city").agg(F.sum("quantity").alias("items"))
                 .orderBy("city"))

def etl(sales, customers_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, customers_big)

try:
    measure(slow_etl(sales, customers_big), "slow", stages=True)
    measure(etl(sales, customers_big), "yours", stages=True)
    etl(sales, customers_big).show()
finally:                                         # back to the defaults, even if etl fails
    for key in ["spark.sql.autoBroadcastJoinThreshold", "spark.sql.adaptive.skewJoin.enabled",
                "spark.sql.adaptive.coalescePartitions.enabled"]:
        spark.conf.unset(key)

## Lesson 6 · Window functions: a group's value on every row

`groupBy` turns each group into **one** row. A **window function** also computes over a group of rows, but it **keeps every row** and writes the result next to it:

```python
w = W.partitionBy("category")                       # the group: rows with the same category
products.withColumn("category_avg", F.avg("price").over(w))
```

* `W.partitionBy(cols)` decides which rows belong together, like `groupBy`. With **no** `partitionBy`, the whole table is one window. That's fine for small data; Spark warns that it moves every row into one partition.
* `.orderBy(cols)` sets the order inside each window. Ranks, `lag`/`lead` and running totals need it (Lessons 7 and 8).
* **Any aggregate** works with `.over(w)`: `sum`, `avg`, `min`, `max`, `count`, `variance` (Zillacode #29)...

Typical uses without an order: compare each row with its group (price minus the category average), a share of the group (`F.col("x") / F.sum("x").over(w)`), the group's size next to each row, and keeping the rows that equal the group's maximum (ties all stay). [Window docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html).

🔬 **How a window runs.** A window needs every row of a partition key in one task, sorted: the plan is `Exchange hashpartitioning(partition key)`, then `Sort` (by the key and the window's `orderBy`), then `Window`. Unlike a `groupBy`, there's **no partial aggregation**: every row is shuffled, not one per key. Several windows with the **same `partitionBy`** share one exchange (a different `orderBy` adds a `Sort`). A window **without `partitionBy`** becomes `Exchange SinglePartition`: the whole table in one task, fine on Zillacode, a disaster on 10 million rows.

A group's total next to every row can also be written as `groupBy` + join back. Which is cheaper? The window shuffles every row once. The `groupBy` shuffles almost nothing (partial aggregation), and its small result can be broadcast back, so the big table isn't shuffled at all, but only if you say so: the planner can't know that an aggregate is small, so without `F.broadcast` it plans a sort-merge join and shuffles the big table anyway. The cells below measure all three.

In [32]:
w = W.partitionBy("category")
products.select(
    "product_name", "category", "price",
    F.round(F.avg("price").over(w), 2).alias("category_avg"),
    F.max("price").over(w).alias("category_max"),
    F.count("*").over(w).alias("in_category"),
    F.round(F.col("price") / F.sum("price").over(w) * 100, 1).alias("pct_of_category"),
).orderBy("category", "price").show()

+------------+-----------+------+------------+------------+-----------+---------------+
|product_name|   category| price|category_avg|category_max|in_category|pct_of_category|
+------------+-----------+------+------------+------------+-----------+---------------+
|       Novel|      Books|  15.0|        20.0|        25.0|          2|           37.5|
|    Cookbook|      Books|  25.0|        20.0|        25.0|          2|           62.5|
|  Headphones|Electronics| 150.0|       675.0|      1200.0|          2|           11.1|
|      Laptop|Electronics|1200.0|       675.0|      1200.0|          2|           88.9|
|   Desk Lamp|       Home|  40.0|        40.0|        40.0|          1|          100.0|
|     Blender|    Kitchen|  60.0|        72.5|        85.0|          2|           41.4|
|Coffee Maker|    Kitchen|  85.0|        72.5|        85.0|          2|           58.6|
+------------+-----------+------+------------+------------+-----------+---------------+



In [33]:
# Keep the rows that equal their group's maximum (ties all stay)
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("max_qty", F.max("quantity").over(W.partitionBy("customer_id")))
       .filter(F.col("quantity") == F.col("max_qty"))
       .select("customer_id", "order_id", "quantity")
       .orderBy("customer_id", "order_id")
       .show())

+-----------+--------+--------+
|customer_id|order_id|quantity|
+-----------+--------+--------+
|          1|    1002|       3|
|          2|    1003|       2|
|          3|    1009|       4|
|          4|    1006|       2|
|          4|    1010|       2|
|          5|    1014|       5|
|          6|    1012|       2|
+-----------+--------+--------+



In [34]:
# Three windows, one partitionBy: one Exchange, two Sorts (one per orderBy), three Window operators
w_date = W.partitionBy("customer_id").orderBy("order_date")
w_qty = W.partitionBy("customer_id").orderBy(F.desc("quantity"))
(orders.withColumn("nth_order", F.row_number().over(w_date))
       .withColumn("qty_rank", F.rank().over(w_qty))
       .withColumn("customer_items", F.sum("quantity").over(W.partitionBy("customer_id")))
       .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [sum(quantity#44) windowspecdefinition(customer_id#42, specifiedwindowframe(RowFrame, unboundedpreceding$(), unboundedfollowing$())) AS customer_items#831L], [customer_id#42]
   +- Window [rank(quantity#44) windowspecdefinition(customer_id#42, quantity#44 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS qty_rank#822], [customer_id#42], [quantity#44 DESC NULLS LAST]
      +- Sort [customer_id#42 ASC NULLS FIRST, quantity#44 DESC NULLS LAST], false, 0
         +- Window [row_number() windowspecdefinition(customer_id#42, order_date#45 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS nth_order#820], [customer_id#42], [order_date#45 ASC NULLS FIRST]
            +- Sort [customer_id#42 ASC NULLS FIRST, order_date#45 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(customer_id#42, 200), ENSURE_REQUIREMENTS, [plan_id=5583]
   

In [35]:
# Each sale's store total, three ways
store_totals = sales.groupBy("store_id").agg(F.sum("quantity").alias("store_items"))
measure(sales.withColumn("store_items", F.sum("quantity").over(W.partitionBy("store_id"))), "window")
measure(sales.join(store_totals, "store_id"), "groupBy + join back")
measure(sales.join(F.broadcast(store_totals), "store_id"), "groupBy + broadcast join back")

window: 2.47 s | 2 jobs, 2 stages, 9 tasks | read 141.9 MB | shuffle 268.3 MB | spill 223.2 MB
groupBy + join back: 1.60 s | 4 jobs, 4 stages, 13 tasks | read 156.5 MB | shuffle 268.4 MB | spill 0 B
groupBy + broadcast join back: 0.59 s | 3 jobs, 3 stages, 9 tasks | read 156.5 MB | shuffle 36.5 KB | spill 0 B


In [36]:
# No partitionBy: all 10 million rows go through one task (Exchange SinglePartition)
measure(sales.withColumn("row_no", F.row_number().over(W.orderBy("sale_id"))), "window without partitionBy", stages=True)

window without partitionBy: 5.35 s | 2 jobs, 2 stages, 5 tasks | read 141.9 MB | shuffle 273.5 MB | spill 288.1 MB
   stage 250:   4 tasks, median 0.69 s, max 0.69 s | read 141.9 MB | shuffle read 0 B, write 273.5 MB
   stage 252:   1 tasks, median 4.63 s, max 4.63 s | read 0 B | shuffle read 273.5 MB, write 0 B


### Exercises

**Q11 · `etl(products)`** Show product_name, category, price and diff_from_avg (the price minus its category's average price, 2 decimals), sorted by category, then price.

Expected output:

```
+------------+-----------+------+-------------+
|product_name|   category| price|diff_from_avg|
+------------+-----------+------+-------------+
|       Novel|      Books|  15.0|         -5.0|
|    Cookbook|      Books|  25.0|          5.0|
|  Headphones|Electronics| 150.0|       -525.0|
|      Laptop|Electronics|1200.0|        525.0|
|   Desk Lamp|       Home|  40.0|          0.0|
|     Blender|    Kitchen|  60.0|        -12.5|
|Coffee Maker|    Kitchen|  85.0|         12.5|
+------------+-----------+------+-------------+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

## Lesson 7 · Ranking: `row_number`, `rank`, `dense_rank`

With an `orderBy` in the window, you can number the rows of each partition:

| quantity | `row_number` | `rank` | `dense_rank` |
|---|---|---|---|
| 5 | 1 | 1 | 1 |
| 3 | 2 | 2 | 2 |
| 3 | 3 | 2 | 2 |
| 1 | 4 | 4 | 3 |

* `F.row_number()`: 1, 2, 3, ..., always unique. Ties get an **arbitrary** order unless you add a tie-breaker column to `orderBy`.
* `F.rank()`: ties share a number, and the next numbers are skipped (Olympic ranking).
* `F.dense_rank()`: ties share a number, nothing is skipped. "Without gaps in the ranking" in a Zillacode text means `dense_rank` (#7).
* `F.ntile(4)` splits each partition into 4 buckets; `F.percent_rank()` goes from 0 to 1.

**Recipes**
* **Top N per group**: rank over `W.partitionBy(group).orderBy(F.desc(value))`, then keep `<= N`. If ties should all stay, use `rank` or `dense_rank`; for exactly N rows, use `row_number` with a tie-breaker.
* **The latest (or first) row per key**, and **removing duplicates while choosing which row stays**: `row_number` over `partitionBy(key).orderBy(F.desc(time))`, then keep `== 1` (Zillacode #13, #30, #37).
* **Numbering the whole table**: `W.orderBy(...)` with no partition (#28, #43).

Always **drop the helper column** (`rn`, `rank`) before returning, unless the output schema asks for it: an extra column fails the checker.

🔬 **Top N runs cheaper than it looks.** When a `row_number`, `rank` or `dense_rank` is followed directly by a filter like `<= N` (Spark 3.5 and later), the plan gets a `WindowGroupLimit`: each task keeps only the top N rows of every key **before** the shuffle (`Partial`), then again after it (`Final`), so only a few rows per key travel. The next cell shows it on the big table.

In [38]:
# Ties: orders 1006 and 1010 both have quantity 2, and five orders have quantity 1
w = W.orderBy(F.desc("quantity"))
w_unique = W.orderBy(F.desc("quantity"), "order_id")     # a tie-breaker makes row_number repeatable
(orders.filter(F.col("customer_id").isin(1, 3, 4))
       .select("order_id", "quantity",
               F.row_number().over(w_unique).alias("row_number"),
               F.rank().over(w).alias("rank"),
               F.dense_rank().over(w).alias("dense_rank"))
       .orderBy("row_number")
       .show())

+--------+--------+----------+----+----------+
|order_id|quantity|row_number|rank|dense_rank|
+--------+--------+----------+----+----------+
|    1009|       4|         1|   1|         1|
|    1002|       3|         2|   2|         2|
|    1006|       2|         3|   3|         3|
|    1010|       2|         4|   3|         3|
|    1001|       1|         5|   5|         4|
|    1004|       1|         6|   5|         4|
|    1007|       1|         7|   5|         4|
|    1015|       1|         8|   5|         4|
|    1016|       1|         9|   5|         4|
+--------+--------+----------+----+----------+



In [39]:
# The FIRST order of each customer: row_number per customer, oldest first, keep 1, drop the helper
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("rn", F.row_number().over(w))
       .filter(F.col("rn") == 1)
       .drop("rn")
       .orderBy("customer_id")
       .show())

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1001|          1|       101|       1|2024-01-10|delivered|
|    1003|          2|       102|       2|2024-01-25|delivered|
|    1004|          3|       103|       1|2024-02-03|cancelled|
|    1006|          4|       104|       2|2024-02-15|  shipped|
|    1008|          5|       101|       1|2024-03-02|  shipped|
|    1012|          6|       103|       2|2024-03-12|  shipped|
+--------+-----------+----------+--------+----------+---------+



In [40]:
# Top 3 sales of each store: WindowGroupLimit (Partial, then Final) around the shuffle
w = W.partitionBy("store_id").orderBy(F.desc("quantity"), "sale_id")
top3 = sales.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") <= 3)
top3.explain()
measure(top3, "top 3 per store")
measure(sales.withColumn("rn", F.row_number().over(w)), "rank every sale")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Filter (rn#949 <= 3)
   +- Window [row_number() windowspecdefinition(store_id#84, quantity#85 DESC NULLS LAST, sale_id#81L ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rn#949], [store_id#84], [quantity#85 DESC NULLS LAST, sale_id#81L ASC NULLS FIRST]
      +- WindowGroupLimit [store_id#84], [quantity#85 DESC NULLS LAST, sale_id#81L ASC NULLS FIRST], row_number(), 3, Final
         +- Sort [store_id#84 ASC NULLS FIRST, quantity#85 DESC NULLS LAST, sale_id#81L ASC NULLS FIRST], false, 0
            +- Exchange hashpartitioning(store_id#84, 200), ENSURE_REQUIREMENTS, [plan_id=6230]
               +- WindowGroupLimit [store_id#84], [quantity#85 DESC NULLS LAST, sale_id#81L ASC NULLS FIRST], row_number(), 3, Partial
                  +- Sort [store_id#84 ASC NULLS FIRST, quantity#85 DESC NULLS LAST, sale_id#81L ASC NULLS FIRST], false, 0
                     +- FileScan parquet [sale_id#8

### Exercises

**Q12 · `etl(orders, products)`** Show the top 2 products of each category by items delivered (on a tie, the smaller product_id first). Columns: category, product_name, items and rank; sorted by category, then rank.

Expected output:

```
+-----------+------------+-----+----+
|   category|product_name|items|rank|
+-----------+------------+-----+----+
|      Books|       Novel|   12|   1|
|      Books|    Cookbook|    1|   2|
|Electronics|  Headphones|    3|   1|
|Electronics|      Laptop|    2|   2|
|    Kitchen|     Blender|    1|   1|
+-----------+------------+-----+----+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

## Lesson 8 · Previous and next rows, running totals

| Expression | Gives |
|---|---|
| `F.lag("x").over(w)` | x of the **previous** row in the window, null for the first row. `F.lag("x", 2)` goes two rows back; `F.lag("x", 1, 0)` gives 0 instead of null. |
| `F.lead("x").over(w)` | x of the **next** row |
| `F.sum("x").over(w.rowsBetween(W.unboundedPreceding, W.currentRow))` | a running total |
| `F.avg("x").over(w.rowsBetween(-2, W.currentRow))` | a moving average of this row and the 2 before it |
| `F.first("x").over(w)`, `F.last("x", ignorenulls=True).over(w)` | the first / last value in the frame |

Typical uses: the change since the previous row (`x - lag(x)`), growth in percent (`(x - lag(x)) / lag(x) * 100`, Zillacode #45), days since the previous order, the previous product of the same customer (#21), detecting a change (`status != lag(status)`), running balances.

**The frame trap.** Once a window has an `orderBy`, an aggregate over it uses a default **frame**: from the first row up to the current row, **plus every row with the same order value** (`rangeBetween(W.unboundedPreceding, W.currentRow)`). So two orders on the same date both get a running total that already includes each other. For a true row-by-row running total, use `rowsBetween(W.unboundedPreceding, W.currentRow)` and a tie-breaker in `orderBy`. For the same reason, `F.last("x").over(w)` with an `orderBy` returns the **current** row's value, because the frame ends there; use `rowsBetween(W.unboundedPreceding, W.unboundedFollowing)` to look at the whole partition. Without an `orderBy`, the frame is always the whole partition (Lesson 6).

🔬 **The frame is in the plan.** `explain()` prints each window's frame: `specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())` is the default frame of an ordered window, which includes the rows with the same order value; `RowFrame` is what `rowsBetween` gives. When a running total looks wrong, the plan tells you which frame Spark used.

In [42]:
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "order_date", "quantity",
    F.lag("quantity").over(w).alias("prev_qty"),
    (F.col("quantity") - F.lag("quantity").over(w)).alias("change"),
    F.lead("order_date").over(w).alias("next_date"),
    F.datediff("order_date", F.lag("order_date").over(w)).alias("days_since_prev"),
).orderBy("order_date", "order_id").show()

+--------+----------+--------+--------+------+----------+---------------+
|order_id|order_date|quantity|prev_qty|change| next_date|days_since_prev|
+--------+----------+--------+--------+------+----------+---------------+
|    1001|2024-01-10|       1|    NULL|  NULL|2024-01-10|           NULL|
|    1002|2024-01-10|       3|       1|     2|2024-02-20|              0|
|    1007|2024-02-20|       1|       3|    -2|2024-03-25|             41|
|    1015|2024-03-25|       1|       1|     0|      NULL|             34|
+--------+----------+--------+--------+------+----------+---------------+



In [43]:
# Orders 1001 and 1002 share a date: the default frame counts them together
w_date = W.partitionBy("customer_id").orderBy("order_date")
w_rows = W.partitionBy("customer_id").orderBy("order_date", "order_id").rowsBetween(W.unboundedPreceding, W.currentRow)
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "order_date", "quantity",
    F.sum("quantity").over(w_date).alias("default_frame"),
    F.sum("quantity").over(w_rows).alias("rows_frame"),
).orderBy("order_date", "order_id").show()

+--------+----------+--------+-------------+----------+
|order_id|order_date|quantity|default_frame|rows_frame|
+--------+----------+--------+-------------+----------+
|    1001|2024-01-10|       1|            4|         1|
|    1002|2024-01-10|       3|            4|         4|
|    1007|2024-02-20|       1|            5|         5|
|    1015|2024-03-25|       1|            6|         6|
+--------+----------+--------+-------------+----------+



In [44]:
# A moving average, and last() with the default frame vs the whole partition
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "quantity",
    F.round(F.avg("quantity").over(w.rowsBetween(-1, W.currentRow)), 2).alias("avg_of_last_2"),
    F.last("order_id").over(w).alias("last_default_frame"),
    F.last("order_id").over(w.rowsBetween(W.unboundedPreceding, W.unboundedFollowing)).alias("last_of_partition"),
).orderBy("order_id").show()

+--------+--------+-------------+------------------+-----------------+
|order_id|quantity|avg_of_last_2|last_default_frame|last_of_partition|
+--------+--------+-------------+------------------+-----------------+
|    1001|       1|          1.0|              1001|             1015|
|    1002|       3|          2.0|              1002|             1015|
|    1007|       1|          2.0|              1007|             1015|
|    1015|       1|          1.0|              1015|             1015|
+--------+--------+-------------+------------------+-----------------+



In [45]:
# The default frame (RangeFrame) vs rowsBetween (RowFrame), as Spark prints them
w = W.partitionBy("customer_id").orderBy("order_date")
orders.select("order_id",
              F.sum("quantity").over(w).alias("default_frame"),
              F.sum("quantity").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)).alias("rows_frame")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#41, default_frame#1051L, rows_frame#1052L]
   +- Window [sum(quantity#44) windowspecdefinition(customer_id#42, order_date#45 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())) AS default_frame#1051L, sum(quantity#44) windowspecdefinition(customer_id#42, order_date#45 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rows_frame#1052L], [customer_id#42], [order_date#45 ASC NULLS FIRST]
      +- Sort [customer_id#42 ASC NULLS FIRST, order_date#45 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(customer_id#42, 200), ENSURE_REQUIREMENTS, [plan_id=6678]
            +- FileScan parquet [order_id#41,customer_id#42,quantity#44,order_date#45] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: str

### Exercises

**Q13 · `etl(orders)`** For every order of customers 1 and 3, show how many days passed since the same customer's previous order (`days_since_prev`, empty for the first order). Show customer_id, order_id, order_date and days_since_prev, sorted by customer_id, then order_date and order_id.

Expected output:

```
+-----------+--------+----------+---------------+
|customer_id|order_id|order_date|days_since_prev|
+-----------+--------+----------+---------------+
|          1|    1001|2024-01-10|           NULL|
|          1|    1002|2024-01-10|              0|
|          1|    1007|2024-02-20|             41|
|          1|    1015|2024-03-25|             34|
|          3|    1004|2024-02-03|           NULL|
|          3|    1009|2024-03-05|             31|
|          3|    1016|2024-03-28|             23|
+-----------+--------+----------+---------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q14 · `etl(orders)`** A running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and `items_so_far` (the items of this order and all earlier orders of the customer). Sort by customer_id, then order_date and order_id.

Expected output:

```
+-----------+--------+----------+--------+------------+
|customer_id|order_id|order_date|quantity|items_so_far|
+-----------+--------+----------+--------+------------+
|          1|    1001|2024-01-10|       1|           1|
|          1|    1002|2024-01-10|       3|           4|
|          1|    1007|2024-02-20|       1|           5|
|          1|    1015|2024-03-25|       1|           6|
|          2|    1003|2024-01-25|       2|           2|
|          2|    1005|2024-02-05|       1|           3|
|          2|    1013|2024-03-18|       1|           4|
+-----------+--------+----------+--------+------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

## A mixed problem, on big data
Like Zillacode's Hard problems: a join, an aggregation and a window together. Write down the steps first.

**Q15 · ⚡ `etl(sales, products_big)`** The top 3 products of each category by items sold: category, rank (1 to 3; on a tie, the smaller product_id first), product_id, product_name and items, sorted by category and rank. Each sale is in `sales` only once (sale_id is unique). `slow_etl` is correct but takes seconds and spills to disk; find its two expensive mistakes in the plan and in `measure`, and fix them.

Expected output:

```
+-----------+----+----------+-------------+-----+
|   category|rank|product_id| product_name|items|
+-----------+----+----------+-------------+-----+
|     Beauty|   1|     41067|Product 41067|  773|
|     Beauty|   2|      5037| Product 5037|  767|
|     Beauty|   3|     25777|Product 25777|  752|
|      Books|   1|      3240| Product 3240|  779|
|      Books|   2|     31590|Product 31590|  754|
|      Books|   3|     44380|Product 44380|  752|
|Electronics|   1|      2961| Product 2961|  771|
|Electronics|   2|     27281|Product 27281|  756|
|Electronics|   3|     44231|Product 44231|  755|
|      Games|   1|     28179|Product 28179|  775|
|      Games|   2|     31579|Product 31579|  770|
|      Games|   3|     36959|Product 36959|  761|
|     Garden|   1|      9465| Product 9465|  778|
|     Garden|   2|     27865|Product 27865|  774|
|     Garden|   3|     36135|Product 36135|  772|
|       Home|   1|     22693|Product 22693|  807|
|       Home|   2|     30933|Product 30933|  783|
|       Home|   3|     47733|Product 47733|  771|
|    Kitchen|   1|     24742|Product 24742|  790|
|    Kitchen|   2|     26492|Product 26492|  767|
|    Kitchen|   3|     48372|Product 48372|  755|
|      Music|   1|     38968|Product 38968|  781|
|      Music|   2|     10298|Product 10298|  780|
|      Music|   3|     31318|Product 31318|  769|
|     Sports|   1|     30516|Product 30516|  768|
|     Sports|   2|     18816|Product 18816|  759|
|     Sports|   3|     21686|Product 21686|  756|
|       Toys|   1|     44304|Product 44304|  786|
|       Toys|   2|     20024|Product 20024|  784|
|       Toys|   3|     24964|Product 24964|  774|
+-----------+----+----------+-------------+-----+
```

**Target:** the same 30 rows, with under 10 MB shuffled and no spill (`slow_etl` shuffles about 500 MB and spills about 1 GB).

In [ ]:
def slow_etl(sales, products_big):
    joined = sales.join(products_big, "product_id").distinct()
    per_product = (joined.groupBy("category", "product_id", "product_name")
                         .agg(F.sum("quantity").alias("items")))
    w = W.partitionBy("category").orderBy(F.desc("items"), "product_id")
    return (per_product.withColumn("rank", F.row_number().over(w))
                       .filter(F.col("rank") <= 3)
                       .select("category", "rank", "product_id", "product_name", "items")
                       .orderBy("category", "rank"))

def etl(sales, products_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, products_big)

measure(slow_etl(sales, products_big), "slow")
measure(etl(sales, products_big), "yours")
etl(sales, products_big).show(30)

## Lesson 9 · Patterns behind the Hard problems

Lessons 6 to 8 already covered top N per group, the latest row per key, running totals and growth. The Hard problems mostly add one of these ideas.

### A. Sessions: split rows into groups wherever there's a gap
"A new session starts when more than 30 minutes passed since the previous event." The recipe:
1. Compute the gap to the previous row with `lag` (per user, ordered by time).
2. Make a flag column: 1 where a new group starts (the gap is too big, or there's no previous row), else 0.
3. A **running sum of the flag** numbers the groups: 1, 1, 1, 2, 2, 3...

```python
w = W.partitionBy("user").orderBy("time")
gap = F.unix_timestamp("time") - F.unix_timestamp(F.lag("time").over(w))           # seconds
flag = F.when(gap.isNull() | (gap > 30 * 60), 1).otherwise(0)
df.withColumn("new", flag).withColumn("session", F.sum("new").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)))
```
After that, `groupBy(user, session)` gives each session's start, end, length and number of events.

In [49]:
# Shopping trips: a new trip starts when more than 30 days passed since the customer's previous order
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
running = w.rowsBetween(W.unboundedPreceding, W.currentRow)
(orders.filter(F.col("customer_id").isin(1, 3))
       .withColumn("gap_days", F.datediff("order_date", F.lag("order_date").over(w)))
       .withColumn("new_trip", F.when(F.col("gap_days").isNull() | (F.col("gap_days") > 30), 1).otherwise(0))
       .withColumn("trip", F.sum("new_trip").over(running))
       .select("customer_id", "order_id", "order_date", "gap_days", "new_trip", "trip")
       .orderBy("customer_id", "order_date", "order_id")
       .show())

+-----------+--------+----------+--------+--------+----+
|customer_id|order_id|order_date|gap_days|new_trip|trip|
+-----------+--------+----------+--------+--------+----+
|          1|    1001|2024-01-10|    NULL|       1|   1|
|          1|    1002|2024-01-10|       0|       0|   1|
|          1|    1007|2024-02-20|      41|       1|   2|
|          1|    1015|2024-03-25|      34|       1|   3|
|          3|    1004|2024-02-03|    NULL|       1|   1|
|          3|    1009|2024-03-05|      31|       1|   2|
|          3|    1016|2024-03-28|      23|       0|   2|
+-----------+--------+----------+--------+--------+----+



### B. Streaks: runs of consecutive days ("gaps and islands")
"The longest run of consecutive days with a login." The trick: number each user's **distinct** days with `row_number`, then subtract that number from the date. Inside a run of consecutive days, the date goes up by 1 and the row number goes up by 1, so `day - row_number` stays **the same**. It changes as soon as a day is skipped. That constant (the "anchor") is a group key:

| day | row_number | day − row_number |
|---|---|---|
| 03-01 | 1 | 02-29 |
| 03-02 | 2 | 02-29 |
| 03-03 | 3 | 02-29 |
| 03-06 | 4 | 03-02 |
| 03-07 | 5 | 03-02 |

Then `groupBy(user, anchor)` gives each streak: `count` is its length, `min` and `max` its first and last day. Take the days `distinct` first: two logins on one day would break the arithmetic.

In [50]:
logins = (spark.createDataFrame(
              [("ann", "2024-03-01"), ("ann", "2024-03-02"), ("ann", "2024-03-02"), ("ann", "2024-03-03"),
               ("ann", "2024-03-06"), ("ann", "2024-03-07"), ("bob", "2024-03-05")], "user string, day string")
          .withColumn("day", F.to_date("day"))
          .distinct())
w = W.partitionBy("user").orderBy("day")
islands = logins.withColumn("rn", F.row_number().over(w)).withColumn("anchor", F.date_sub("day", F.col("rn")))
islands.orderBy("user", "day").show()
(islands.groupBy("user", "anchor")
        .agg(F.min("day").alias("start"), F.max("day").alias("end"), F.count("*").alias("days"))
        .orderBy("user", "start")
        .show())

+----+----------+---+----------+
|user|       day| rn|    anchor|
+----+----------+---+----------+
| ann|2024-03-01|  1|2024-02-29|
| ann|2024-03-02|  2|2024-02-29|
| ann|2024-03-03|  3|2024-02-29|
| ann|2024-03-06|  4|2024-03-02|
| ann|2024-03-07|  5|2024-03-02|
| bob|2024-03-05|  1|2024-03-04|
+----+----------+---+----------+

+----+----------+----------+----------+----+
|user|    anchor|     start|       end|days|
+----+----------+----------+----------+----+
| ann|2024-02-29|2024-03-01|2024-03-03|   3|
| ann|2024-03-02|2024-03-06|2024-03-07|   2|
| bob|2024-03-04|2024-03-05|2024-03-05|   1|
+----+----------+----------+----------+----+



### C. The first and the last row of each group, in one `groupBy`
`F.min_by(value, ordering)` returns `value` from the row with the smallest `ordering`, and `F.max_by` from the row with the largest. Both work inside a normal `agg`, next to other aggregates. To break ties, order by a **struct**: `F.struct("order_date", "order_id")` compares the date first and the id second. The alternative is two `row_number` windows (one ascending, one descending) and a join, which is longer.

In [51]:
by_time = F.struct("order_date", "order_id")
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.count("*").alias("orders"),
            F.min_by("order_id", by_time).alias("first_order"),
            F.max_by("order_id", by_time).alias("last_order"),
            F.max_by("status", by_time).alias("last_status"))
       .orderBy("customer_id")
       .show())

+-----------+------+-----------+----------+-----------+
|customer_id|orders|first_order|last_order|last_status|
+-----------+------+-----------+----------+-----------+
|          1|     4|       1001|      1015|    shipped|
|          2|     3|       1003|      1013|  delivered|
|          3|     3|       1004|      1016|  delivered|
|          4|     2|       1006|      1010|  cancelled|
|          5|     2|       1008|      1014|  delivered|
|          6|     1|       1012|      1012|    shipped|
+-----------+------+-----------+----------+-----------+



### D. The latest version of every record from several snapshots
Two (or more) copies of a table taken at different times, and you need the current state, including records that exist in only one copy:
1. Tag each copy with its date: `withColumn("snapshot", F.lit(...))`.
2. `unionByName` them.
3. `row_number` over `partitionBy(key).orderBy(F.desc("snapshot"))`, then keep 1.

The same recipe handles a single table with repeated keys and an "updated_at" column (Zillacode #30 removes duplicated keys this way).

### E. Comparing every row with the whole table: three ways
"Products more expensive than the average", "rides whose rating is far from the overall average" (#32). The overall value can come from:
1. **A Python value**: `avg = df.agg(F.avg("x")).first()[0]`, then `F.lit(avg)`. Short, but it runs a separate job.
2. **A cross join with a one-row aggregate**: `df.crossJoin(df.agg(F.avg("x").alias("avg_x")))`. Stays in one plan.
3. **A window over the whole table**: `F.avg("x").over(W.partitionBy())`. No join, but every row goes to one partition.

All three give the same answer; on Zillacode's small inputs, choose whichever reads best.

In [52]:
avg_price = products.agg(F.avg("price")).first()[0]
one = products.select("product_name", "price", F.lit(avg_price).alias("avg_all"))
two = products.crossJoin(products.agg(F.avg("price").alias("avg_all"))).select("product_name", "price", "avg_all")
three = products.select("product_name", "price", F.avg("price").over(W.partitionBy()).alias("avg_all"))
three.orderBy("product_name").show()
print("same result all three ways:", sorted(one.collect()) == sorted(two.collect()) == sorted(three.collect()))

+------------+------+-------+
|product_name| price|avg_all|
+------------+------+-------+
|     Blender|  60.0|  225.0|
|Coffee Maker|  85.0|  225.0|
|    Cookbook|  25.0|  225.0|
|   Desk Lamp|  40.0|  225.0|
|  Headphones| 150.0|  225.0|
|      Laptop|1200.0|  225.0|
|       Novel|  15.0|  225.0|
+------------+------+-------+

same result all three ways: True


### Exercises

**Q16 · `etl(events)`** Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than 30 minutes passed since the customer's previous event. Show customer_id, event_time, page and session (1, 2, ... per customer), sorted by customer_id, then event_time.

Expected output:

```
+-----------+-------------------+-------+-------+
|customer_id|         event_time|   page|session|
+-----------+-------------------+-------+-------+
|          1|2024-03-01 09:00:00|   home|      1|
|          1|2024-03-01 09:10:00|product|      1|
|          1|2024-03-01 11:00:00|   home|      2|
|          1|2024-03-02 10:00:00|   cart|      3|
|          1|2024-03-03 10:30:00|   home|      4|
|          1|2024-03-05 08:00:00|   home|      5|
|          2|2024-03-01 20:00:00|   home|      1|
|          2|2024-03-01 20:20:00|product|      1|
|          2|2024-03-01 20:45:00|   cart|      1|
|          2|2024-03-04 18:00:00|   home|      2|
|          2|2024-03-05 18:00:00|   home|      3|
+-----------+-------------------+-------+-------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

**Q17 · `etl(events)`** What is each customer's longest streak of consecutive days with at least one event? Show customer_id and longest_streak, sorted by customer_id.

Expected output:

```
+-----------+--------------+
|customer_id|longest_streak|
+-----------+--------------+
|          1|             3|
|          2|             2|
|          3|             3|
+-----------+--------------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

**Q18 · `etl(customers, customers_new)`** Build the current customer list from two snapshots: `customers` (taken on 2024-03-31) and `customers_new` (taken on 2024-04-07). Keep the newest row of every customer, including customers found only in the older snapshot. Show customer_id, name, city and snapshot (the date of that row's snapshot), sorted by customer_id.

Expected output:

```
+-----------+----------------+--------+----------+
|customer_id|            name|    city|  snapshot|
+-----------+----------------+--------+----------+
|          1|    Aziz Karimov|Tashkent|2024-04-07|
|          2|   Bella Schmidt|  Munich|2024-04-07|
|          3|     Carlos Ruiz|  Madrid|2024-04-07|
|          4|Dilnoza Rahimova|Tashkent|2024-04-07|
|          5|     Emma Wilson|  London|2024-04-07|
|          6|    Farid Aliyev|    NULL|2024-04-07|
|          7|       Grace Lee|  Berlin|2024-03-31|
|          8|       Hana Sato|   Osaka|2024-04-07|
+-----------+----------------+--------+----------+
```

In [ ]:
def etl(customers, customers_new):
    # Write code here: return a DataFrame
    return customers

etl(customers, customers_new).show()

## Lesson 10 · Spark SQL inside `etl`

Zillacode's template creates `spark`, so you can answer in SQL whenever that's easier for you:
1. Register each input DataFrame as a temporary view: `orders.createOrReplaceTempView("orders")`.
2. Return `spark.sql("SELECT ...")`, which is a DataFrame like any other.

Everything you have learned so far has a SQL form: `CASE WHEN ... THEN ... ELSE ... END`, `COALESCE`, `GROUP BY ... HAVING`, every join type (`LEFT ANTI JOIN`, `LEFT SEMI JOIN`, `CROSS JOIN`), window functions (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`), `date_format`, `datediff`... You can also mix the two styles: `F.expr("...")` puts a piece of SQL inside DataFrame code.

Two habits: alias every computed column (`AS revenue`), and filter a window result in an outer query (a `WHERE` can't see a window column of the same `SELECT`). [SQL reference](https://spark.apache.org/docs/latest/sql-ref-syntax.html).

🔬 **In the plan.** `spark.sql(...)` and DataFrame code go through the same Catalyst optimizer and produce the **same physical plan** (next cell). Choose the style that reads best for each problem; it never changes the speed. The same goes for `F.expr("...")` and `selectExpr`.

In [56]:
def etl(orders, products):
    orders.createOrReplaceTempView("orders")
    products.createOrReplaceTempView("products")
    return spark.sql("""
        SELECT p.category, SUM(o.quantity * p.price) AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'delivered'
        GROUP BY p.category
        ORDER BY revenue DESC
    """)

etl(orders, products).show()

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 2850.0|
|      Books|  205.0|
|    Kitchen|   60.0|
+-----------+-------+



In [57]:
# A window function in SQL, filtered in an outer query
products.createOrReplaceTempView("products")
spark.sql("""
    SELECT category, product_name, price, price_rank
    FROM (SELECT category, product_name, price,
                 RANK() OVER (PARTITION BY category ORDER BY price DESC) AS price_rank
          FROM products)
    WHERE price_rank = 1
    ORDER BY category
""").show()

+-----------+------------+------+----------+
|   category|product_name| price|price_rank|
+-----------+------------+------+----------+
|      Books|    Cookbook|  25.0|         1|
|Electronics|      Laptop|1200.0|         1|
|       Home|   Desk Lamp|  40.0|         1|
|    Kitchen|Coffee Maker|  85.0|         1|
+-----------+------------+------+----------+



In [58]:
# The same query in SQL and with DataFrame methods: the physical plans are the same
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
spark.sql("""
    SELECT p.category, SUM(o.quantity) AS items
    FROM orders o JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'delivered'
    GROUP BY p.category
""").explain()
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .groupBy("category").agg(F.sum("quantity").alias("items"))
       .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category#39], functions=[sum(quantity#44)])
   +- Exchange hashpartitioning(category#39, 200), ENSURE_REQUIREMENTS, [plan_id=9410]
      +- HashAggregate(keys=[category#39], functions=[partial_sum(quantity#44)])
         +- Project [quantity#44, category#39]
            +- BroadcastHashJoin [product_id#43], [product_id#37], Inner, BuildLeft, false, false
               :- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=9405]
               :  +- Project [product_id#43, quantity#44]
               :     +- Filter ((isnotnull(status#46) AND (status#46 = delivered)) AND isnotnull(product_id#43))
               :        +- FileScan parquet [product_id#43,quantity#44,status#46] Batched: true, DataFilters: [isnotnull(status#46), (status#46 = delivered), isnotnull(product_id#43)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab

### Exercise

**Q19 · `etl(products)`** Solve it with `spark.sql`: the cheapest product of each category. Show category, product_name and price, sorted by category.

Expected output:

```
+-----------+------------+-----+
|   category|product_name|price|
+-----------+------------+-----+
|      Books|       Novel| 15.0|
|Electronics|  Headphones|150.0|
|       Home|   Desk Lamp| 40.0|
|    Kitchen|     Blender| 60.0|
+-----------+------------+-----+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

## Lesson 11 · 🔬 Read less: columns, row groups, folders and files

The cheapest data is the data Spark never reads. Four levers (theory Day 4), measured on `sales` with the `read` number of `measure` (the bytes read from files):

**1. Columns.** Parquet stores each column separately, so a query that uses 2 of the 7 columns reads only those two (column pruning, Day 13). `select("*")` before a write, or a cache of the whole table, reads every column.

**2. Row groups, when the data is clustered.** A filter pushed into the scan (Day 13, Lesson 2) skips the row groups whose min/max can't match. But in `sales` the dates are random: every row group holds dates from the whole two years, so a one-month filter can't skip anything. Written **sorted by date**, each file covers a narrow date range, and the same filter skips almost all of them. Sort the data you write by the column you filter on most (`orderBy`, or `repartitionByRange` plus `sortWithinPartitions`).

**3. Partition folders.** `df.write.partitionBy("sale_month")` writes one folder per value: `sale_month=2024-12/`. A filter on that column becomes `PartitionFilters` in the plan, and Spark lists and opens only the matching folders. The column disappears from the files and comes back from the folder names ([partition discovery](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#partition-discovery)). Partition by a column you filter on often that has **few values** (a month, a country). A column with millions of values (customer_id) makes millions of tiny folders and files.

**4. File size.** Every file costs a task fixed work: list it, open it, read its footer. A read task takes up to `spark.sql.files.maxPartitionBytes` (128 MB) of files, counting `spark.sql.files.openCostInBytes` (4 MB) for opening each one, so thousands of small files mean many more tasks and much more overhead before any data is read ([file source options](https://spark.apache.org/docs/latest/sql-data-sources-generic-options.html)). Aim for files of roughly 100 MB to 1 GB, and choose the number of output files before writing:

| Method | Plan | Shuffle? | Use it to |
|---|---|---|---|
| `df.coalesce(n)` | `Coalesce n` | no | merge partitions into fewer, bigger ones (fewer output files). It can't add partitions, and the work before it runs in only n tasks |
| `df.repartition(n)` | `Exchange RoundRobinPartitioning(n)` | yes | spread the rows evenly over n partitions |
| `df.repartition(n, "col")` | `Exchange hashpartitioning(col, n)` | yes | put all rows of a key in one partition (and one file) |
| `df.repartitionByRange(n, "col")` | `Exchange rangepartitioning(col, n)` | yes | sorted ranges, e.g. files that each hold a range of dates |

In [60]:
# 1. Columns: 2 of 7 columns, then all of them
measure(sales.select("sale_id", "customer_id"), "2 columns")
measure(sales, "all 7 columns")

2 columns: 0.11 s | 1 jobs, 1 stages, 4 tasks | read 72.5 MB | shuffle 0 B | spill 0 B
all 7 columns: 0.27 s | 1 jobs, 1 stages, 4 tasks | read 141.9 MB | shuffle 0 B | spill 0 B


In [61]:
# 2. Row groups: the same December filter on files in random date order, then on files sorted by date
import os
if not os.path.exists("data/big/sales_by_date/_SUCCESS"):
    sales.orderBy("sale_date").write.mode("overwrite").parquet("data/big/sales_by_date")
sales_by_date = spark.read.parquet("data/big/sales_by_date")
december = lambda df: df.filter(F.col("sale_date").between("2024-12-01", "2024-12-31")).agg(F.sum("quantity"))
measure(december(sales), "December, random order")
measure(december(sales_by_date), "December, sorted by date")

December, random order: 0.09 s | 2 jobs, 2 stages, 5 tasks | read 17.6 MB | shuffle 236 B | spill 0 B
December, sorted by date: 0.04 s | 2 jobs, 2 stages, 6 tasks | read 373.1 KB | shuffle 287 B | spill 0 B


In [62]:
# 3. Partition folders: one folder per month, and a filter on the folder column
if not os.path.exists("data/big/sales_by_month/_SUCCESS"):
    (sales.withColumn("sale_month", F.date_format("sale_date", "yyyy-MM"))
          .write.mode("overwrite").partitionBy("sale_month").parquet("data/big/sales_by_month"))
folders = sorted(d for d in os.listdir("data/big/sales_by_month") if d.startswith("sale_month="))
print(len(folders), "folders:", folders[:3], "...")
by_month = spark.read.parquet("data/big/sales_by_month")
dec_folder = by_month.filter(F.col("sale_month") == "2024-12").agg(F.sum("quantity"))
dec_folder.explain()
measure(dec_folder, "December folder only")

24 folders: ['sale_month=2023-01', 'sale_month=2023-02', 'sale_month=2023-03'] ...
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(quantity#1516)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=9716]
      +- HashAggregate(keys=[], functions=[partial_sum(quantity#1516)])
         +- Project [quantity#1516]
            +- FileScan parquet [quantity#1516,sale_month#1519] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales_by_month], PartitionFilters: [isnotnull(sale_month#1519), (sale_month#1519 = 2024-12)], PushedFilters: [], ReadSchema: struct<quantity:int>


December folder only: 0.08 s | 2 jobs, 2 stages, 5 tasks | read 290.7 KB | shuffle 236 B | spill 0 B


In [63]:
# 4. Small files: the same table as 500 files
if not os.path.exists("data/big/sales_500_files/_SUCCESS"):
    sales.repartition(500).write.mode("overwrite").parquet("data/big/sales_500_files")
many_files = spark.read.parquet("data/big/sales_500_files")
print("read partitions:", many_files.rdd.getNumPartitions(), "for 500 files vs", sales.rdd.getNumPartitions(), "for 8")
measure(many_files.agg(F.sum("quantity")), "sum, 500 files")
measure(sales.agg(F.sum("quantity")), "sum, 8 files")

read partitions: 17 for 500 files vs 4 for 8
sum, 500 files: 0.14 s | 2 jobs, 2 stages, 18 tasks | read 23.8 MB | shuffle 1.0 KB | spill 0 B
sum, 8 files: 0.04 s | 2 jobs, 2 stages, 5 tasks | read 4.3 MB | shuffle 236 B | spill 0 B


In [64]:
# coalesce merges partitions without a shuffle; repartition shuffles
sales.coalesce(2).explain()
sales.repartition(8, "store_id").explain()

== Physical Plan ==
Coalesce 2
+- *(1) ColumnarToRow
   +- FileScan parquet [sale_id#81L,customer_id#82,product_id#83,store_id#84,quantity#85,sale_date#86,channel#87] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:date,channel:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange hashpartitioning(store_id#84, 8), REPARTITION_BY_NUM, [plan_id=9928]
   +- FileScan parquet [sale_id#81L,customer_id#82,product_id#83,store_id#84,quantity#85,sale_date#86,channel#87] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:date,channel:stri

### Exercises

**Q20 · 🔍 Read the plan** **Predict** which of the three queries reads the fewest bytes and which the most, then run the cell. Why does (b) read so much less than (a), although its plan has no `PartitionFilters`?

In [ ]:
dec_dates = lambda df: df.filter(F.col("sale_date").between("2024-12-01", "2024-12-31")).agg(F.sum("quantity"))
measure(dec_dates(sales), "(a) sales, filter on sale_date")
measure(dec_dates(by_month), "(b) month folders, filter on sale_date")
measure(by_month.filter(F.col("sale_month") == "2024-12").agg(F.sum("quantity")), "(c) month folders, filter on sale_month")
dec_dates(by_month).explain()
# Your answer:

**Q21 · ⚡ `etl(by_month)`** Add up the items of each channel in the first quarter of 2024 (January to March), sorted by channel. `slow_etl` gets it right but reads every month folder. Make Spark open only three of them.

Expected output:

```
+-------+-------+
|channel|  items|
+-------+-------+
|    app|1246124|
|  store|1243699|
|    web|1245358|
+-------+-------+
```

**Target:** the plan's `PartitionFilters` select the three months, and `read` drops from about 14 MB to about 1 MB.

In [ ]:
def slow_etl(by_month):
    return (by_month.filter((F.year("sale_date") == 2024) & (F.quarter("sale_date") == 1))
                    .groupBy("channel").agg(F.sum("quantity").alias("items"))
                    .orderBy("channel"))

def etl(by_month):
    # Write code here: return a DataFrame
    return slow_etl(by_month)

measure(slow_etl(by_month), "slow")
measure(etl(by_month), "yours")
etl(by_month).show()
etl(by_month).explain()

## Lesson 12 · 🔬 Caching: compute once, use several times

A DataFrame is a recipe, not data: every action runs the whole recipe again from the files. Three reports built on one expensive DataFrame compute it three times. `df.cache()` keeps the computed rows the first time an action needs them, and later actions read them from there (theory Day 7; [caching in the tuning guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html#caching-data)).

* `cache()` is **lazy**: it only marks the DataFrame. The first action fills the cache, and it's the slowest one: it computes everything *and* stores it.
* For DataFrames, `cache()` is `persist(StorageLevel.MEMORY_AND_DISK)`: in memory, and on disk what doesn't fit. `persist(StorageLevel.DISK_ONLY)` and the other levels trade memory for speed ([persist docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.persist.html)).
* In the plan, the cached part becomes an `InMemoryRelation` read by `InMemoryTableScan`. The Spark UI's **Storage** tab lists every cached DataFrame with its size in memory and on disk.
* `df.unpersist()` frees it (`spark.catalog.clearCache()` frees everything). Until then it stays, unless Spark needs the memory for running tasks.
* A cached DataFrame keeps the partitions it had when it was cached. AQE can't coalesce them afterwards, so a join or aggregate result cached with 200 partitions is read by 200 tasks.

**It pays when** the same DataFrame is used by several actions, it's expensive to compute (a join, a big aggregation, a window), and what you store is small enough to fit in memory: an aggregate is ideal.

**It hurts when** the DataFrame is used once (you pay to store it for nothing), or when you cache a raw table and then filter it: the cache holds every row and column, and the filter can no longer reach the Parquet reader (last cell). And `cache()` inside a loop without `unpersist()` fills the memory.

**Cache, checkpoint or a file?** A cache keeps the lineage: if an executor is lost, Spark recomputes the missing partitions. `df.checkpoint()` writes the data to reliable storage and cuts the lineage, which helps very long plans (loops that build on their own result). Writing an intermediate result to Parquet is the most robust choice: another job can read it tomorrow.

In [67]:
# An expensive DataFrame (a full pass over 10 million sales, with a shuffle) that comes out small: one row per customer
per_customer = (sales.groupBy("customer_id")
                     .agg(F.count("*").alias("purchases"), F.sum("quantity").alias("items"),
                          F.min("sale_date").alias("first_sale"), F.max("sale_date").alias("last_sale")))
def three_reports():
    per_customer.groupBy("purchases").count().collect()                            # customers by number of purchases
    per_customer.orderBy(F.desc("items")).limit(10).collect()                       # the top 10 customers
    per_customer.groupBy(F.date_format("first_sale", "yyyy-MM")).count().collect()   # new customers per month
measure(three_reports, "3 reports, no cache")
per_customer.cache()
measure(lambda: per_customer.count(), "fill the cache")
measure(three_reports, "3 reports from the cache")
for r in ui("storage/rdd"):                     # what the Storage tab shows
    print("cached:", size(r["memoryUsed"]), "in memory,", size(r["diskUsed"]), "on disk,",
          r["numCachedPartitions"], "of", r["numPartitions"], "partitions")

3 reports, no cache: 3.87 s | 8 jobs, 8 stages, 26 tasks | read 126.5 MB | shuffle 173.4 MB | spill 0 B
fill the cache: 1.85 s | 4 jobs, 4 stages, 405 tasks | read 69.9 MB | shuffle 90.3 MB | spill 0 B
3 reports from the cache: 1.57 s | 5 jobs, 5 stages, 602 tasks | read 62.1 MB | shuffle 470.0 KB | spill 0 B
cached: 20.7 MB in memory, 0 B on disk, 200 of 200 partitions


In [68]:
# In the plan, the reports read the cache (InMemoryTableScan) instead of the files
per_customer.groupBy("purchases").count().explain()
print("still cached:", per_customer.unpersist().is_cached)      # unpersist() frees the memory

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[purchases#1687L], functions=[count(1)])
   +- Exchange hashpartitioning(purchases#1687L, 200), ENSURE_REQUIREMENTS, [plan_id=10892]
      +- HashAggregate(keys=[purchases#1687L], functions=[partial_count(1)])
         +- InMemoryTableScan [purchases#1687L]
               +- InMemoryRelation [customer_id#82, purchases#1687L, items#1688L, first_sale#1689, last_sale#1690], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- AdaptiveSparkPlan isFinalPlan=true
                        +- == Final Plan ==
                           ResultQueryStage 1
                           +- *(2) HashAggregate(keys=[customer_id#82], functions=[count(1), sum(quantity#85), min(sale_date#86), max(sale_date#86)])
                              +- ShuffleQueryStage 0
                                 +- Exchange hashpartitioning(customer_id#82, 200), ENSURE_REQUIREMENTS, [plan_id=10713]
                      

In [69]:
# Caching the raw table, then filtering: every row is cached, and the filter can't reach the Parquet reader
sales.cache()
sales.filter(F.col("store_id") == 7).explain()
print("still cached:", sales.unpersist().is_cached)

== Physical Plan ==
*(1) Filter (isnotnull(store_id#84) AND (store_id#84 = 7))
+- InMemoryTableScan [sale_id#81L, customer_id#82, product_id#83, store_id#84, quantity#85, sale_date#86, channel#87], [isnotnull(store_id#84), (store_id#84 = 7)]
      +- InMemoryRelation [sale_id#81L, customer_id#82, product_id#83, store_id#84, quantity#85, sale_date#86, channel#87], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) ColumnarToRow
               +- FileScan parquet [sale_id#81L,customer_id#82,product_id#83,store_id#84,quantity#85,sale_date#86,channel#87] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:date,channel:string>


still cached: False


### Exercise

**Q22 · 🔍 Think it through** Which of these four programs gain from `cache()`, and which lose? Answer without running them.

In [ ]:
# (a) big = sales.join(customers_big, "customer_id").cache()
#     print(big.count())
#
# (b) per_store = sales.groupBy("store_id", "channel").agg(F.sum("quantity").alias("items")).cache()
#     then 5 different reports are built from per_store
#
# (c) sales.cache()
#     sales.filter(F.col("store_id") == 7).show()
#
# (d) clean = (an expensive cleaning of sales: joins, windows).cache()
#     for month in the 24 months: clean.filter(month).write.parquet(one folder per month)
#
# Your answers:

## Lesson 13 · UDFs: a Python function as a column function

`F.udf(python_function, return_type)` turns a plain Python function into a column function. Some official Zillacode solutions use one (#31 checks weekends with Python's `datetime`, #40 counts list items with Python's `split`). It works, but:
* A UDF runs Python for every row, outside Spark's optimizer, so it's slower than built-in functions (theory Day 8).
* **The function itself must handle `None`**: Spark passes null as `None`, and `None.split(",")` crashes the job.
* Almost every case has a built-in: `F.dayofweek(F.to_date(col, "MM/dd/yyyy")).isin(1, 7)` for weekends, `F.size(F.split(...))` with a null guard for counting.

Use a UDF only when no built-in function (or combination of them) can do the job. [udf docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html).

🔬 **What a UDF costs.** Spark can't look inside a Python function, so: (1) **every row** is sent from the JVM to a Python worker process and the result comes back; (2) a condition that uses a UDF can't be pushed into the scan (`PushedFilters` stays empty), and the UDF breaks the stage's generated code; (3) Python runs much slower than the JVM code of built-in functions. The plan shows where rows leave the JVM: `BatchEvalPython` (rows pickled in batches, the default in Spark 3 and so on Zillacode) or `ArrowEvalPython` (rows in Arrow column batches: the default for `F.udf` in Spark 4.2, and always for `pandas_udf`). The choice is made when the UDF is created: `F.udf(f, "string", useArrow=False)` forces pickled rows. The next cells measure one rule four ways on 10 million rows. Expect the built-in to win, the pickled UDF to lose, and Arrow and pandas UDFs in between; the first Python UDF of a session also pays for starting the Python workers, hence the warm-up. [Arrow and pandas UDFs](https://spark.apache.org/docs/latest/api/python/tutorial/sql/arrow_pandas.html).

In [71]:
import datetime                                   # Zillacode's template imports the module like this
from pyspark.sql.types import IntegerType, BooleanType

count_items = F.udf(lambda s: len(s.split(",")) if s else 0, IntegerType())        # handles None and ""
is_weekend = F.udf(lambda s: datetime.datetime.strptime(s, "%m/%d/%Y").weekday() >= 5 if s else None, BooleanType())

t = spark.createDataFrame([("a,b,c", "03/02/2024"), ("", "03/04/2024"), (None, None)], "items string, day string")
t.select(
    "items", "day",
    count_items("items").alias("count_udf"),
    F.when(F.col("items").isNull() | (F.col("items") == ""), 0).otherwise(F.size(F.split("items", ","))).alias("count_builtin"),
    is_weekend("day").alias("weekend_udf"),
    F.dayofweek(F.to_date("day", "MM/dd/yyyy")).isin(1, 7).alias("weekend_builtin"),
).show()

+-----+----------+---------+-------------+-----------+---------------+
|items|       day|count_udf|count_builtin|weekend_udf|weekend_builtin|
+-----+----------+---------+-------------+-----------+---------------+
|a,b,c|03/02/2024|        3|            3|       true|           true|
|     |03/04/2024|        0|            0|      false|          false|
| NULL|      NULL|        0|            0|       NULL|           NULL|
+-----+----------+---------+-------------+-----------+---------------+



In [72]:
# One rule (big = 4 items or more) four ways. The Arrow choice is fixed when a UDF is created:
# useArrow=True is Spark 4.2's default for F.udf; useArrow=False pickles rows, like Spark 3 (and Zillacode)
import numpy as np, pandas as pd
from pyspark.sql.functions import pandas_udf
band = lambda q: "big" if q >= 4 else "small"
udf_arrow = F.udf(band, "string", useArrow=True)
udf_pickled = F.udf(band, "string", useArrow=False)
@pandas_udf("string")
def band_pandas(q: pd.Series) -> pd.Series:          # gets a whole batch as a pandas Series
    return pd.Series(np.where(q >= 4, "big", "small"))

measure(sales.limit(10_000).select(udf_arrow("quantity"), udf_pickled("quantity"), band_pandas("quantity")),
        "warm-up: starts the Python workers")
measure(sales.select("sale_id", F.when(F.col("quantity") >= 4, "big").otherwise("small").alias("band")), "built-in")
measure(sales.select("sale_id", udf_arrow("quantity").alias("band")), "udf, Arrow")
measure(sales.select("sale_id", udf_pickled("quantity").alias("band")), "udf, pickled")
measure(sales.select("sale_id", band_pandas("quantity").alias("band")), "pandas_udf")

warm-up: starts the Python workers: 0.55 s | 2 jobs, 2 stages, 5 tasks | read 2.1 MB | shuffle 86.9 KB | spill 0 B
built-in: 0.21 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B
udf, Arrow: 1.06 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B
udf, pickled: 1.42 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B
pandas_udf: 0.60 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B


In [73]:
# The plans: ArrowEvalPython, BatchEvalPython, and the built-in condition, which is the only one pushed into the scan
sales.filter(udf_arrow("quantity") == "big").select("sale_id").explain()
sales.filter(udf_pickled("quantity") == "big").select("sale_id").explain()
sales.filter(F.col("quantity") >= 4).select("sale_id").explain()

== Physical Plan ==
*(2) Project [sale_id#81L]
+- *(2) Filter (pythonUDF0#2548 = big)
   +- ArrowEvalPython [<lambda>(quantity#85)#2547], [pythonUDF0#2548], 101
      +- *(1) ColumnarToRow
         +- FileScan parquet [sale_id#81L,quantity#85] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,quantity:int>


== Physical Plan ==
*(2) Project [sale_id#81L]
+- *(2) Filter (pythonUDF0#2550 = big)
   +- BatchEvalPython [<lambda>(quantity#85)#2549], [pythonUDF0#2550]
      +- *(1) ColumnarToRow
         +- FileScan parquet [sale_id#81L,quantity#85] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,quantity:int>


== Physical Plan ==
*(1) Project [sale_id#81L]
+- *(1) Filter (isnotnull(quan

## Lesson 14 · Debugging: errors, and the mistakes that give no error

**The errors you will meet**, and what they mean:

| Message (start) | Cause | Fix |
|---|---|---|
| `[UNRESOLVED_COLUMN.WITH_SUGGESTION] ... cannot be resolved. Did you mean ...` | a typo, or a column that an earlier `select`/`groupBy` removed | read the suggested names; `print(df.columns)` |
| `[AMBIGUOUS_REFERENCE]` | both sides of a join have that column | Lesson 3 |
| `[CANNOT_CONVERT_COLUMN_INTO_BOOL]` | `and` / `or` / `not` / `if` used on a column | `&`, `\|`, `~`, `F.when` |
| `Column is not iterable` | Python's own `sum`, `max`, `len` used on a column | `F.sum`, `F.max`, `F.length` |
| `'Column' object is not callable` | a DataFrame method called on a column (`F.col("x").count()`) | `F.count("x")` inside `agg` |
| `'NoneType' object has no attribute ...`, or Zillacode's *etl function must return ... not NoneType* | the function returns nothing, or returns `df.show()` (which prints and returns `None`) | `return df` |
| `[DATATYPE_MISMATCH]` | e.g. `F.size` on a text, or `union` of incompatible columns | check the types with `df.printSchema()` |

**Mistakes that give no error, just a wrong answer.** Check these before you press Run:
1. **Columns**: exactly the output schema's names; no helper columns (`rn`, `gap`, `flag`) left over.
2. **Types**: an `avg` or a `/` gives a double; if the schema says Integer, `cast("int")`. Round only when the task says so, and to the stated number of decimals.
3. **Nulls**: filters with `!=` drop null rows; `count("*")` vs `count(col)`; left joins need `coalesce(..., 0)` when the task wants 0.
4. **Joins**: "all X" means a left join; two detail tables joined before aggregating multiply each other (the fan-out trap); null keys never match.
5. **Ties**: `row_number` (exactly N rows) vs `rank` (gaps) vs `dense_rank` (no gaps), and a tie-breaker where the task names one.
6. **Duplicates**: `distinct` before counting days or pages; `count_distinct` when the task says "different" or "unique".
7. **Dates**: text input means `to_date`/`to_timestamp` with the right pattern (`MM` months, `mm` minutes).
8. **Hidden test**: no values copied from the example (ids, dates, counts).

### 🔬 Performance mistakes that give no error

| Mistake | Why it's slow | Instead |
|---|---|---|
| `df.collect()` or `df.toPandas()` on a big table | every row travels to the driver, one process on one machine; it can fail with an out-of-memory error or `spark.driver.maxResultSize` | aggregate or `limit` first; write big results to files |
| `if df.count() > 0:` | counts every row to answer yes or no | `if not df.isEmpty():` stops at the first row |
| `df = df.withColumn(...)` in a loop over hundreds of columns | each call analyzes a bigger plan on the driver, before any data is read | one `select(...)` or `withColumns({...})` |
| the same expensive DataFrame in several actions, without a cache | each action computes it again | `cache()` it (Lesson 12) |
| `for row in df.collect():` with Spark calls inside the loop | one Spark job per row | one join or `groupBy` |
| `show()` to check that a query is fast | `show()` runs only enough of the plan to print 20 rows (`CollectLimit`) | `measure()`, or a write |
| `coalesce(1)` / `repartition(1)` before writing a big table | the whole table goes through one task (and `coalesce(1)` pulls the work before it into that task too) | let Spark write several files of a sensible size |
| an `orderBy` nobody needs | a full shuffle and sort (Day 13, Q1) | sort only the final, small result |

In [74]:
# A typo: the message suggests the real names
try:
    orders.select("order_dte")
except Exception as e:
    print(type(e).__name__, "->", str(e).split("SQLSTATE")[0][:230])

AnalysisException -> [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `order_dte` cannot be resolved. Did you mean one of the following? [`order_date`, `order_id`, `product_id`, `quantity`, `status`]. 


In [75]:
# show() prints and returns None: never assign or return it
result = orders.filter(F.col("quantity") > 3).show()
print("result is", result)

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1009|          3|       105|       4|2024-03-05|delivered|
|    1014|          5|       105|       5|2024-03-20|delivered|
+--------+-----------+----------+--------+----------+---------+

result is None


In [76]:
# Python's own functions on a column, and a DataFrame method on a column
for attempt in [lambda: sum(F.col("quantity")), lambda: F.col("quantity").count()]:
    try:
        attempt()
    except Exception as e:
        print(type(e).__name__, "->", str(e).split(".")[0][:120])

PySparkTypeError -> [NOT_ITERABLE] Column is not iterable
TypeError -> 'Column' object is not callable


In [77]:
# Is there a big order? count() reads every row; isEmpty() stops at the first one
big_orders = sales.filter(F.col("quantity") == 5)
measure(lambda: big_orders.count() > 0, "count() > 0")
measure(lambda: big_orders.isEmpty(), "isEmpty()")

count() > 0: 0.09 s | 2 jobs, 2 stages, 5 tasks | read 5.0 MB | shuffle 236 B | spill 0 B
isEmpty(): 0.03 s | 1 jobs, 2 stages, 5 tasks | read 2.5 MB | shuffle 184 B | spill 0 B


In [78]:
# 200 new columns: a withColumn loop vs one withColumns call. Only the plan is built here, nothing runs
start = time.perf_counter()
df = sales
for i in range(200):
    df = df.withColumn(f"c{i}", F.col("quantity") + i)
print(f"200 x withColumn: {time.perf_counter() - start:.2f} s")
start = time.perf_counter()
df = sales.withColumns({f"c{i}": F.col("quantity") + i for i in range(200)})
print(f"1 x withColumns:  {time.perf_counter() - start:.2f} s")

200 x withColumn: 1.15 s
1 x withColumns:  0.18 s


## The slow report: a final optimization problem

A real job, as it often looks before anyone reads its plan. It returns the right numbers, but it makes four of the mistakes you've met in Days 13 to 15. Find them in `explain()` and `measure`, then fix them.

**Q23 · ⚡ `etl(sales, customers_big)`** For 2024, show for each city: sales (the number of sales), items, customers (the number of different customers) and weekend_pct (the percentage of the city's sales made on a Saturday or Sunday, 1 decimal), sorted by city. `slow_etl` returns exactly this. Write a version without its four problems.

Expected output:

```
+---------+-------+-------+---------+-----------+
|     city|  sales|  items|customers|weekend_pct|
+---------+-------+-------+---------+-----------+
|   Berlin| 350551|1050390|   165371|       28.4|
|  Bukhara| 351129|1054126|   165230|       28.4|
|   London| 350391|1052052|   165255|       28.4|
|   Madrid| 350491|1051790|   165527|       28.5|
|   Munich| 350579|1051998|   165400|       28.4|
|    Osaka| 350956|1053709|   165289|       28.4|
|    Paris| 350578|1050359|   165432|       28.4|
|     Rome| 351569|1054367|   165730|       28.5|
|Samarkand|1850566|5550118|   165340|       28.4|
| Tashkent| 351930|1056961|   165424|       28.5|
+---------+-------+-------+---------+-----------+
```

**Target:** no `ArrowEvalPython`/`BatchEvalPython`, no `InMemoryTableScan`, a `BroadcastHashJoin`, and the date filter in `PushedFilters`. `measure` should be well under half the slow time. (The cell clears Spark's cache first, so the slow version is measured from scratch every time.)

In [ ]:
def slow_etl(sales, customers_big):
    is_weekend = F.udf(lambda d: d.weekday() >= 5, "boolean")
    in_2024 = sales.filter(F.year("sale_date") == 2024)
    joined = in_2024.join(customers_big, "customer_id").cache()
    per_city = joined.groupBy("city").agg(F.count("*").alias("sales"), F.sum("quantity").alias("items"))
    buyers = joined.groupBy("city").agg(F.countDistinct("customer_id").alias("customers"))
    weekend = joined.filter(is_weekend("sale_date")).groupBy("city").agg(F.count("*").alias("weekend_sales"))
    return (per_city.join(buyers, "city").join(weekend, "city")
                    .withColumn("weekend_pct", F.round(F.col("weekend_sales") / F.col("sales") * 100, 1))
                    .select("city", "sales", "items", "customers", "weekend_pct")
                    .orderBy("city"))

def etl(sales, customers_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, customers_big)

spark.catalog.clearCache()                       # measure the slow version from scratch
measure(slow_etl(sales, customers_big), "slow")
spark.catalog.clearCache()
measure(etl(sales, customers_big), "yours")
etl(sales, customers_big).show()
etl(sales, customers_big).explain()
spark.catalog.clearCache()

## Optimization checklist

What Days 13 to 15 taught, as questions to ask of any slow job (and in interviews):

| Ask | Look at | Fix |
|---|---|---|
| What does Spark actually run? | `explain()`, the SQL / DataFrame tab | read it bottom-up; count the `Exchange`s |
| Does it read only what it needs? | `ReadSchema`, `PushedFilters`, `PartitionFilters`, `read` in `measure` | select columns; filter on raw columns (no functions around them); partition folders by a filter column; sort the files you write |
| How many shuffles, and of how much? | `Exchange` operators; shuffle write per stage | one `agg` per key; aggregate before joining; no `distinct` or `orderBy` that isn't needed; top N with a filter right after the rank |
| Which join algorithm? | `BroadcastHashJoin` vs `SortMergeJoin`, `BroadcastNestedLoopJoin` | `F.broadcast` small, filtered or aggregated sides; add an equality to range joins |
| Is one task much slower than the others? | median vs max in the Stages tab, `measure(stages=True)` | broadcast, AQE skew join, salting, a separate path for the hot key |
| Does a window have a `partitionBy`? | `Exchange SinglePartition` | always partition; a `groupBy` + broadcast join back for group totals |
| Is Python on the path of every row? | `BatchEvalPython`, `ArrowEvalPython` | built-in functions |
| Is the same thing computed twice? | the same scans and joins twice in the plan | one pass; `cache()` what several actions reuse, then `unpersist()` |
| Does the driver do the work? | `collect`, `toPandas`, `count() > 0`, loops | aggregate in Spark, `isEmpty()`, one `select` |
| Did it actually get faster? | `measure` or the UI, before and after | change one thing at a time |

Official guides to keep: [Performance tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html), [Tuning Spark](https://spark.apache.org/docs/latest/tuning.html), [Web UI](https://spark.apache.org/docs/latest/web-ui.html), [Configuration](https://spark.apache.org/docs/latest/configuration.html).

## Next: Zillacode

You now know every tool its 52 PySpark problems use, and how Spark runs them. Set it up (the last task in the roadmap), then solve the problems Easy first, then Medium, then Hard. For each one: read the output schema, plan the steps in a comment, write `etl`, and check the "no error, wrong answer" list from Lesson 14 before you press Run. If a problem fails, compare your output with the expected one row by row; the difference almost always points at nulls, ties, types or an extra column.

Zillacode builds its inputs from Python dicts with `spark.createDataFrame`, so the columns come in **alphabetical order**, whole numbers are `bigint`, decimals are `double`, and **times and dates are text** (convert them with `to_date` or `to_timestamp`).

Zillacode's problems by topic (number and level: E, M, H), so you can drill a weak topic:

| Topic | Problems |
|---|---|
| Complex Joins | 2M, 3H, 5E, 10M, 13H, 14E, 15H, 18M, 20H, 23M, 24H, 27M, 31M, 38E, 46E, 47E, 49M, 52E |
| Aggregate Functions | 3H, 5E, 8E, 10M, 15H, 19M, 20H, 23M, 25E, 27M, 32M, 35E, 36E, 39E, 42E, 44M |
| Windows Functions | 7H, 9M, 13H, 15H, 21H, 22H, 24H, 28M, 29H, 30H, 37H, 43H, 45H |
| Regular Expressions | 4M, 6E, 17E, 26H, 30H, 51H |
| Conditional Logic | 1E, 7H, 32M, 36E, 50E |
| Datetime Operations | 1E, 20H, 22H, 30H, 31M |
| Case When Statements | 16E, 21H, 33M, 41E, 45H |
| String Manipulation | 2M, 21H, 40M, 48M |
| Null Handling | 11E, 14E, 26H |
| Mathematical Functions | 29H, 45H, 51H |
| Distinct Function | 8E, 13H |
| Union Statements | 12E, 34E |
| Pivot Function | 3H |

## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Which customers have never placed an order? Show customer_id and name.</summary>

```python
def etl(customers, orders):
    return customers.join(orders, "customer_id", "left_anti").select("customer_id", "name").orderBy("customer_id")

etl(customers, orders).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;Show every customer with their number of orders, including customers with 0 orders. Show name and orders, ...</summary>

```python
def etl(customers, orders):
    return (customers.join(orders, "customer_id", "left")
                     .groupBy("customer_id", "name")
                     .agg(F.count("order_id").alias("orders"))
                     .select("name", "orders")
                     .orderBy(F.col("orders").desc(), "name"))

etl(customers, orders).show()
```

`count("order_id")` counts only matched rows, so Grace gets 0; `count("*")` would give her 1.

</details>

<details>
<summary><b>Q3</b> &nbsp;Which customers had all of their orders delivered (and at least one order)? Show their names, from A to Z.</summary>

```python
def etl(orders, customers):
    not_delivered = orders.filter(F.col("status") != "delivered").select("customer_id")
    return (customers.join(orders, "customer_id", "left_semi")
                     .join(not_delivered, "customer_id", "left_anti")
                     .select("name")
                     .orderBy("name"))

etl(orders, customers).show()
```

`left_semi` keeps customers with at least one order (once each), and `left_anti` then removes those with an order that isn't delivered.

</details>

<details>
<summary><b>Q4</b> &nbsp;Predict the algorithm of each join, and which side is broadcast.</summary>

(a) `BroadcastHashJoin ... LeftOuter, BuildRight`: customers, the right side, is broadcast. A left join can only build its right side.

(b) `BroadcastHashJoin ... LeftAnti, BuildRight`: the same rule for `left_anti`. Only `customer_id` is read from customers: an anti join needs nothing else from that side.

(c) `SortMergeJoin ... FullOuter`, with an `Exchange` and a `Sort` on both sides: a full join can't broadcast, not even 7 rows.

(d) `BroadcastNestedLoopJoin BuildLeft`: there's no `=` in the condition, so neither a hash nor a sort-merge join is possible. Spark broadcasts one side (for an inner join it may build either; here products, the side it estimated smaller) and compares every product with every promotion.

</details>

<details>
<summary><b>Q5</b> &nbsp;For every city and segment, count the sales and add up the items, over all 10 million sales. Sort by city ...</summary>

```python
def etl(sales, customers_big):
    return (sales.join(F.broadcast(customers_big), "customer_id")
                 .groupBy("city", "segment")
                 .agg(F.count("*").alias("sales"), F.sum("quantity").alias("items"))
                 .orderBy("city", "segment"))

etl(sales, customers_big).show()
```

The query needs three columns of `customers_big` (`customer_id`, `city`, `segment`), which Spark estimates at about 13 MB: over the 10 MB limit, so it plans a sort-merge join and shuffles every sale, then spills while sorting the reseller's 3 million rows. The hint makes each task build a hash table of the 2 million customers and stream its sales through it; only the small partial aggregates are shuffled. The alternative `spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50MB")` works too, but it changes every join of the session. Lesson 6 of Day 13 suggests aggregating sales per customer before the join: that helps less here, because 10 million sales still come down to about 1.9 million customers.

</details>

<details>
<summary><b>Q6</b> &nbsp;Show every referral with both names: referral_date, referrer (the name of the customer who invited) and ...</summary>

```python
def etl(referrals, customers):
    referrer = customers.select(F.col("customer_id").alias("referrer_id"), F.col("name").alias("referrer"))
    referred = customers.select(F.col("customer_id").alias("referred_id"), F.col("name").alias("referred"))
    return (referrals.join(referrer, "referrer_id", "left")
                     .join(referred, "referred_id", "left")
                     .select("referral_date", "referrer", "referred")
                     .orderBy("referral_date"))

etl(referrals, customers).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Show the pairs of customers who live in the same city: city, customer_a and customer_b (names). Each pair ...</summary>

```python
def etl(customers):
    a, b = customers.alias("a"), customers.alias("b")
    return (a.join(b, (F.col("a.city") == F.col("b.city")) & (F.col("a.customer_id") < F.col("b.customer_id")))
             .select(F.col("a.city").alias("city"), F.col("a.name").alias("customer_a"), F.col("b.name").alias("customer_b"))
             .orderBy("city"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;For each delivered order, find the promotion of its product's category that was running on the order date. ...</summary>

```python
def etl(orders, products, promotions):
    return (orders.filter(F.col("status") == "delivered")
                  .join(products, "product_id")
                  .join(promotions, (products.category == promotions.category)
                                    & F.col("order_date").between(F.col("start_date"), F.col("end_date")))
                  .select("order_id", "product_name", "promo_id", "discount_pct")
                  .orderBy("order_id"))

etl(orders, products, promotions).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;A window partitioned by customer_id: which stage is skewed, why doesn't AQE fix it, and what would you do?</summary>

The window's stage (200 tasks): a median around 0.02 s and a slowest task around 3 s, over 100 times the median. That task holds customer 1's partition: 3 million rows to receive, sort and number, while every other task has about 35,000.

AQE can't fix it: its skew handling only splits the partitions of a **join**. A window needs every row of a key in one task, in order, so the partition of one key can't be split.

For a daily job: (1) check whether the reseller belongs in this report at all, and if not, filter it out before the window; (2) if you only need one value per customer (a count, a total, the first date), use `groupBy`: partial aggregation makes the hot key cheap (the last cell of Lesson 5); (3) otherwise give the hot key its own path (sort its 3 million rows with `orderBy`, which spreads them over many tasks by range), run the window on the other customers, and `union` the two; (4) at least give the tasks enough memory not to spill.

</details>

<details>
<summary><b>Q10</b> &nbsp;Add up the items of each city, sorted by city. In this cell Spark can't broadcast and AQE's skew join is off ...</summary>

```python
def etl(sales, customers_big):
    salts = 16
    # big side: customer 1's rows get a random salt 0..15, every other row gets 0
    salted_sales = sales.withColumn(
        "salt", F.when(F.col("customer_id") == 1, (F.rand(42) * salts).cast("int")).otherwise(0))
    # other side: customer 1's row is copied once per salt value, every other row once with salt 0
    salted_customers = customers_big.withColumn("salt", F.explode(
        F.when(F.col("customer_id") == 1, F.array(*[F.lit(i) for i in range(salts)]))
         .otherwise(F.array(F.lit(0)))))
    return (salted_sales.join(salted_customers, ["customer_id", "salt"])
                        .groupBy("city").agg(F.sum("quantity").alias("items"))
                        .orderBy("city"))

etl(sales, customers_big).show()
```

The join key is now (customer_id, salt). Customer 1's 3 million sales are spread over 16 salt values, about 190,000 rows each, which hash into different partitions, and each of those partitions finds its copy of customer 1's row. Every other customer has salt 0 on both sides, so it joins as before, and the totals don't change. Only the hot key is copied (16 rows instead of 1); salting every key would copy all 2 million customers 16 times. The gain is smaller than 16 times because the 16 salted keys are hashed into 200 partitions, and two or three of them often land in the same one. With only `customer_id` and `quantity` shuffled from sales, the reseller's partition is about 10 MB compressed, so even with lowered thresholds AQE wouldn't treat it as skewed (Lesson 5): this is the kind of case where salting by hand is still needed. In real life, try `F.broadcast` first.

Why the total time hardly changes here: with 4 cores, the other 199 tasks keep three cores busy while the hot task runs anyway. On a cluster, where hundreds of cores wait for one straggler, the slowest task decides the stage's time, and that's what salting fixes.

</details>

<details>
<summary><b>Q11</b> &nbsp;Show product_name, category, price and diff_from_avg (the price minus its category's average price, 2 ...</summary>

```python
def etl(products):
    w = W.partitionBy("category")
    return (products.select("product_name", "category", "price",
                            F.round(F.col("price") - F.avg("price").over(w), 2).alias("diff_from_avg"))
                    .orderBy("category", "price"))

etl(products).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;Show the top 2 products of each category by items delivered (on a tie, the smaller product_id first). ...</summary>

```python
def etl(orders, products):
    items = (orders.filter(F.col("status") == "delivered")
                   .join(products, "product_id")
                   .groupBy("category", "product_id", "product_name")
                   .agg(F.sum("quantity").alias("items")))
    w = W.partitionBy("category").orderBy(F.col("items").desc(), "product_id")
    return (items.withColumn("rank", F.row_number().over(w))
                 .filter(F.col("rank") <= 2)
                 .select("category", "product_name", "items", "rank")
                 .orderBy("category", "rank"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;For every order of customers 1 and 3, show how many days passed since the same customer's previous order ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isin(1, 3))
                  .withColumn("days_since_prev", F.datediff("order_date", F.lag("order_date").over(w)))
                  .select("customer_id", "order_id", "order_date", "days_since_prev")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;A running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and items_so_far ...</summary>

```python
def etl(orders):
    w = (W.partitionBy("customer_id").orderBy("order_date", "order_id")
          .rowsBetween(W.unboundedPreceding, W.currentRow))
    return (orders.filter(F.col("customer_id").isin(1, 2))
                  .withColumn("items_so_far", F.sum("quantity").over(w))
                  .select("customer_id", "order_id", "order_date", "quantity", "items_so_far")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q15</b> &nbsp;The top 3 products of each category by items sold: category, rank (1 to 3; on a tie, the smaller product_id ...</summary>

```python
def etl(sales, products_big):
    per_product = sales.groupBy("product_id").agg(F.sum("quantity").alias("items"))
    w = W.partitionBy("category").orderBy(F.desc("items"), "product_id")
    return (per_product.join(products_big, "product_id")
                       .withColumn("rank", F.row_number().over(w))
                       .filter(F.col("rank") <= 3)
                       .select("category", "rank", "product_id", "product_name", "items")
                       .orderBy("category", "rank"))

etl(sales, products_big).show(30)
```

Mistake 1: `distinct()` "just in case". The rows are already unique (sale_id), but Spark doesn't know that, so it shuffles all 10 million joined rows with every column to compare them (about 500 MB, most of the spill). Mistake 2: joining before aggregating. The join makes 10 million wide rows, and the `groupBy` then hashes the text columns `category` and `product_name` for each of them. Aggregating `sales` by `product_id` first shrinks 10 million rows to 50,000 inside each task (partial aggregation), and only then does the broadcast join add the names. The window at the end is cheap either way: it ranks 50,000 rows, with a `WindowGroupLimit` before its shuffle.

</details>

<details>
<summary><b>Q16</b> &nbsp;Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than ...</summary>

```python
def etl(events):
    w = W.partitionBy("customer_id").orderBy("event_time")
    gap_minutes = (F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60
    return (events.dropDuplicates()
                  .filter(F.col("customer_id").isin(1, 2))
                  .withColumn("new_session", F.when(gap_minutes.isNull() | (gap_minutes > 30), 1).otherwise(0))
                  .withColumn("session", F.sum("new_session").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)))
                  .select("customer_id", "event_time", "page", "session")
                  .orderBy("customer_id", "event_time"))

etl(events).show()
```

</details>

<details>
<summary><b>Q17</b> &nbsp;What is each customer's longest streak of consecutive days with at least one event? Show customer_id and ...</summary>

```python
def etl(events):
    days = events.select("customer_id", F.to_date("event_time").alias("day")).distinct()
    w = W.partitionBy("customer_id").orderBy("day")
    streaks = (days.withColumn("anchor", F.date_sub("day", F.row_number().over(w)))
                   .groupBy("customer_id", "anchor")
                   .agg(F.count("*").alias("days")))
    return (streaks.groupBy("customer_id")
                   .agg(F.max("days").alias("longest_streak"))
                   .orderBy("customer_id"))

etl(events).show()
```

</details>

<details>
<summary><b>Q18</b> &nbsp;Build the current customer list from two snapshots: customers (taken on 2024-03-31) and customers_new (taken ...</summary>

```python
def etl(customers, customers_new):
    both = (customers.withColumn("snapshot", F.to_date(F.lit("2024-03-31")))
                     .unionByName(customers_new.withColumn("snapshot", F.to_date(F.lit("2024-04-07")))))
    w = W.partitionBy("customer_id").orderBy(F.desc("snapshot"))
    return (both.withColumn("rn", F.row_number().over(w))
                .filter(F.col("rn") == 1)
                .select("customer_id", "name", "city", "snapshot")
                .orderBy("customer_id"))

etl(customers, customers_new).show()
```

</details>

<details>
<summary><b>Q19</b> &nbsp;Solve it with spark.sql: the cheapest product of each category. Show category, product_name and price, sorted ...</summary>

```python
def etl(products):
    products.createOrReplaceTempView("products")
    return spark.sql("""
        SELECT category, product_name, price
        FROM (SELECT category, product_name, price,
                     ROW_NUMBER() OVER (PARTITION BY category ORDER BY price) AS rn
              FROM products)
        WHERE rn = 1
        ORDER BY category
    """)

etl(products).show()
```

</details>

<details>
<summary><b>Q20</b> &nbsp;Which of three December queries reads the fewest bytes, and why does the month-folder table help even without PartitionFilters?</summary>

From the most bytes to the fewest: (a) about 17.6 MB, (b) about 1.4 MB, (c) about 290 KB.

(a) The files are in random date order: every row group's min and max of `sale_date` span the two years, so no row group can be skipped, and both columns the query needs (`sale_date` and `quantity`) are read in full.

(b) There are no `PartitionFilters` (the filter is on `sale_date`, not on the folder column), so Spark lists every folder and opens every file. But each file holds the dates of one month, so the min and max of every row group outside December rule it out: the reader reads its footer and skips its data. The `PushedFilters` on `sale_date` do this work.

(c) `PartitionFilters` on `sale_month`: Spark doesn't even list the other 23 folders; it opens only December's files. Lesson: partition by the column you filter on, and filter on the partition column itself.

</details>

<details>
<summary><b>Q21</b> &nbsp;Add up the items of each channel in the first quarter of 2024 (January to March), sorted by channel. slow_etl ...</summary>

```python
def etl(by_month):
    return (by_month.filter(F.col("sale_month").between("2024-01", "2024-03"))
                    .groupBy("channel").agg(F.sum("quantity").alias("items"))
                    .orderBy("channel"))

etl(by_month).show()
```

`sale_month` is text like `2024-01`, so `between("2024-01", "2024-03")` (or `isin("2024-01", "2024-02", "2024-03")`) compares text, which works because the format sorts like the dates. The slow version wraps `sale_date` in `year()` and `quarter()`: nothing can be pushed, and every folder and file is read. A filter on the raw dates, `F.col("sale_date").between("2024-01-01", "2024-03-31")`, would skip most row groups through their min/max (as in (b) of Q20), but every file's footer is still read; only a filter on the partition column skips whole folders.

</details>

<details>
<summary><b>Q22</b> &nbsp;Which of four programs gain from cache(), and which lose?</summary>

**Gain: (b) and (d).** (b) is the ideal case: an expensive pass over 10 million rows that comes out small (at most 600 rows) and is reused five times. (d) reuses the expensive result 24 times; without the cache, each write would redo the joins and windows (also consider `clean.write.partitionBy("month")`, a single write that does it in one pass).

**Lose: (a) and (c).** (a) uses the DataFrame once: the cache only adds the cost of storing 10 million joined rows with every column. (c) stores all 10 million sales, every column, to show 20 rows of one store; and the filter can no longer be pushed into the Parquet scan (Lesson 12's last cell). `sales.filter(...).show()` alone reads far less.

</details>

<details>
<summary><b>Q23</b> &nbsp;For 2024, show for each city: sales (the number of sales), items, customers (the number of different ...</summary>

```python
def etl(sales, customers_big):
    weekend = F.dayofweek("sale_date").isin(1, 7)
    return (sales.filter(F.col("sale_date").between("2024-01-01", "2024-12-31"))
                 .join(F.broadcast(customers_big), "customer_id")
                 .groupBy("city")
                 .agg(F.count("*").alias("sales"),
                      F.sum("quantity").alias("items"),
                      F.countDistinct("customer_id").alias("customers"),
                      F.sum(F.when(weekend, 1).otherwise(0)).alias("weekend_sales"))
                 .withColumn("weekend_pct", F.round(F.col("weekend_sales") / F.col("sales") * 100, 1))
                 .select("city", "sales", "items", "customers", "weekend_pct")
                 .orderBy("city"))

etl(sales, customers_big).show()
```

The four problems of `slow_etl`:
1. **A Python UDF** for the weekend: `ArrowEvalPython` in the plan, every 2024 sale sent to Python. `F.dayofweek(...).isin(1, 7)` stays in the JVM (Lesson 13).
2. **`F.year("sale_date") == 2024`** wraps the column, so `PushedFilters` has no date (Day 13, Lesson 2). On these random-order files it reads the same bytes either way, but on files sorted or partitioned by date (Lesson 11) it decides how much is read.
3. **The same joined data aggregated three times and joined back** (Day 13, Lesson 5): one `groupBy` with all four aggregates does it in one pass; the weekend count becomes a conditional sum.
4. **A `cache()` that only exists because of problem 3**, and it makes things worse: the cache must hold every column of the join, so column pruning stops there. `customers_big` is then needed with all four columns, about 18 MB, which is over the broadcast limit: a sort-merge join shuffles 5 million sales. Without the cache, only `customer_id` and `city` are needed (about 8 MB), and the join is broadcast (`F.broadcast` makes it explicit).

`countDistinct` still costs a second shuffle (by city and customer); if an estimate within a few percent were acceptable, `approx_count_distinct` would remove it, but the task asks for the exact number.

</details>